# DigiHero: Sample Description and Usage Patterns

This notebook contains the analysis workflow for the assessing the demographic and clinical information and usage patterns of the DigiHero sample. It is organized to mirror the manuscript structure (sample definition → descriptives → psychometrics → AI usage patterns → downstream consequences).

# Notebook roadmap

- **Sample inclusion / missingness**
- **Analysis sample exclusions**
- **Descriptive statistics** (demographics and clinical variables)
- **Psychometric scale scores**
- **AI usage** (patterns, tools, and use cases)
- **Consequences and side effects**
- **Trust in AI answers**


In [1]:
# -*- coding: utf-8 -*-
# =============================================================================
# Sample overview: AI use (Yes/No)
# =============================================================================
# Purpose: Descriptive distribution of AI use among participants with non-missing
#          age and gender variables used for the analysis sample definition.
# Input:   DATA_FILE (CSV/Excel export; semicolon-separated CSV expected)
# Output:  Printed counts, percentages, and raw value counts for the AI-use item.
# =============================================================================

import sys
import pandas as pd
import numpy as np
from scipy import stats

# Windows terminal UTF-8 fix (safe to keep on all platforms)
if hasattr(sys.stdout, 'reconfigure'):
    sys.stdout.reconfigure(encoding='utf-8', errors='replace')

# -----------------------------------------------------------------------------
# LOAD DATA  -- change filename here
# -----------------------------------------------------------------------------
DATA_FILE = 'KIdaten_renamed.csv'   # change to your filename

if DATA_FILE.endswith('.csv'):
    # sep=';'          : German/European CSV exports use semicolons
    # on_bad_lines='skip' : skips rows with free-text commas/newlines
    df = pd.read_csv(DATA_FILE, sep=';', low_memory=False, on_bad_lines='skip')
else:
    df = pd.read_excel(DATA_FILE)

# Uses df from cell above; KI question variable is G1Q1
ki_col = 'ki_nutzen_sie_ki_programme_wie_z_b_chatgpt_gemini_claud'

mask = df['gender_filled'].notna() & df['birth_year_filled'].notna()
df_sub = df.loc[mask].copy()

yes_labels = ['Ja', 'ja', 'Yes', 'yes']
no_labels  = ['Nein', 'nein', 'No', 'no']
n_yes   = df_sub[ki_col].astype(str).str.strip().isin(yes_labels).sum()
n_no    = df_sub[ki_col].astype(str).str.strip().isin(no_labels).sum()
n_other = len(df_sub) - n_yes - n_no
n       = len(df_sub)

print('KI usage (only rows with gender_filled and birth_year_filled):')
print(f'  Yes:   {n_yes:>6}  ({100*n_yes/n:.1f}%)')
print(f'  No:    {n_no:>6}  ({100*n_no/n:.1f}%)')
print(f'  Other: {n_other:>6}  ({100*n_other/n:.1f}%)')
print(f'  Total N: {n}')
print()
print('Value counts (raw):')
print(df_sub[ki_col].value_counts(dropna=False))

KI usage (only rows with gender_filled and birth_year_filled):
  Yes:    15058  (51.2%)
  No:     13852  (47.1%)
  Other:    524  (1.8%)
  Total N: 29434

Value counts (raw):
ki_nutzen_sie_ki_programme_wie_z_b_chatgpt_gemini_claud
Ja      15058
Nein    13852
NaN       524
Name: count, dtype: int64


## Sample inclusion: exposure to the clinical questionnaire

This section quantifies how many participants were **shown** the clinical questionnaire (i.e., were presented the relevant items), independent of whether they completed all items.


In [2]:
# How many rows have any non-missing value in lastpage_mds?
# (counts as missing: NaN, empty string, whitespace-only)

col = 'lastpage_mds'

missing = df[col].isna() | (df[col].astype(str).str.strip() == '')
has_value = ~missing

n_total = len(df)
n_with_value = int(has_value.sum())
n_missing = int(missing.sum())

print(f"Rows with a value in {col}: {n_with_value} ({100 * n_with_value / n_total:.1f}%)")
print(f"Rows missing {col}:          {n_missing} ({100 * n_missing / n_total:.1f}%)")
print(f"Total rows:                  {n_total}")

Rows with a value in lastpage_mds: 13186 (44.6%)
Rows missing lastpage_mds:          16385 (55.4%)
Total rows:                  29571


## Missingness assessment: sociodemographics

We compare **complete cases** versus participants with missingness in key sociodemographic variables.


In [3]:
import pandas as pd

# Same definitions as your exclusion / descriptives cells
ki_col = 'ki_nutzen_sie_ki_programme_wie_z_b_chatgpt_gemini_claud'
gender_col = 'gender_filled'
age_col = 'birth_year_filled'
employed_col = 'sozdem_sind_sie_zurzeit_erwerbstaetig'
school_col = 'sozdem_welchen_hoechsten_allgemeinbildenden_schulabschl'
edu_coded_col = 'education_level_coded'

extra_cols = [employed_col, school_col, edu_coded_col]


def is_blank(s: pd.Series) -> pd.Series:
    """Missing = NaN or empty / whitespace-only string (matches your KI/gender/age logic)."""
    return s.isna() | (s.astype(str).str.strip() == '')


# Answered AI question (non-blank)
answered_ai = ~is_blank(df[ki_col])
n_answered_ai = int(answered_ai.sum())

# Among those: non-missing each field
df_ai = df.loc[answered_ai]

miss_gender = is_blank(df_ai[gender_col])
miss_age = is_blank(df_ai[age_col])
miss_employed = is_blank(df_ai[employed_col])
miss_school = is_blank(df_ai[school_col])
miss_edu_coded = is_blank(df_ai[edu_coded_col])

complete_demo = ~miss_gender & ~miss_age
complete_all = complete_demo & ~miss_employed & ~miss_school & ~miss_edu_coded

n_complete = int(complete_all.sum())
n_incomplete = int((~complete_all).sum())

print("Among participants who answered the AI usage question:")
print(f"  N (answered AI):     {n_answered_ai}")
print(f"  N complete on gender, age, employment, school degree, education_level_coded: {n_complete} "
      f"({100 * n_complete / n_answered_ai:.1f}%)")
print(f"  N missing ≥1 of those: {n_incomplete} ({100 * n_incomplete / n_answered_ai:.1f}%)")
print()

print("Missing counts within AI-answerers (each column separately):")
for name, col, m in [
    ("gender_filled", gender_col, miss_gender),
    ("birth_year_filled", age_col, miss_age),
    ("employment", employed_col, miss_employed),
    ("school degree", school_col, miss_school),
    ("education_level_coded", edu_coded_col, miss_edu_coded),
]:
    print(f"  {name:<32} missing: {int(m.sum()):>6}  "
          f"non-missing: {int((~m).sum()):>6}")

Among participants who answered the AI usage question:
  N (answered AI):     29045
  N complete on gender, age, employment, school degree, education_level_coded: 28507 (98.1%)
  N missing ≥1 of those: 538 (1.9%)

Missing counts within AI-answerers (each column separately):
  gender_filled                    missing:     99  non-missing:  28946
  birth_year_filled                missing:    125  non-missing:  28920
  employment                       missing:    447  non-missing:  28598
  school degree                    missing:    404  non-missing:  28641
  education_level_coded            missing:    416  non-missing:  28629


In [4]:
import numpy as np
import pandas as pd

# --- Columns ---
ki_col = 'ki_nutzen_sie_ki_programme_wie_z_b_chatgpt_gemini_claud'
gender_col = 'gender_filled'
age_col = 'birth_year_filled'
employed_col = 'sozdem_sind_sie_zurzeit_erwerbstaetig'
school_col = 'sozdem_welchen_hoechsten_allgemeinbildenden_schulabschl'
edu_coded_col = 'education_level_coded'
edu_school_col = school_col

phq4_cols = ['PHQ4_1', 'PHQ4_2', 'PHQ4_3', 'PHQ4_4']
diagnose_col = 'Diagnose_selbst_dich'
eq5d_col = 'EQ5D5L_3'
behandlung_col = 'Behandlung_dich'

REFERENCE_YEAR = 2026


def is_blank(s: pd.Series) -> pd.Series:
    return s.isna() | (s.astype(str).str.strip() == '')


def is_binary_male_female(s: pd.Series) -> pd.Series:
    g = s.astype(str).str.strip().str.lower()
    return g.isin(['männlich', 'maennlich']) | g.eq('weiblich')


def employment_valid_ja_nein(s: pd.Series) -> pd.Series:
    """
    Same idea as KI_Regression1: answered with Ja… vs Nein… / No…
    Invalid = blank, NaN, or any other text.
    """
    t = s.astype(str).str.strip()
    blank = s.isna() | t.eq('') | t.str.lower().eq('nan')
    emp_yes = t.str.startswith('Ja', na=False) | t.str.startswith('ja', na=False)
    emp_no = (
        t.str.startswith('Nein', na=False)
        | t.str.startswith('nein', na=False)
        | t.str.startswith('No', na=False)
        | t.str.startswith('no', na=False)
    )
    return ~blank & (emp_yes | emp_no)


def miss_phq4_item(s: pd.Series) -> pd.Series:
    n = pd.to_numeric(s, errors='coerce')
    return ~n.isin([1, 2, 3, 4])


def miss_lime_letter(s: pd.Series, valid: set[str]) -> pd.Series:
    v = s.astype(str).str.strip().str.lower()
    blank = s.isna() | v.eq('') | v.isin(['nan', 'none', '<na>'])
    return blank | ~v.isin(valid)


# --- Base: answered AI + gender + age ---
answered_ai = ~is_blank(df[ki_col])
has_demo = answered_ai & ~is_blank(df[gender_col]) & ~is_blank(df[age_col])
df_base = df.loc[has_demo].copy()
n_base = len(df_base)

# --- Education → years (on full df) ---
school_years_map = {
    "Allgemeine oder fachgebundene Hochschulreife/Abitur (Gymnasium bzw. Abschluss der Erweiterten Oberschule (der DDR))": 13,
    "Fachhochschulreife, Abschluss einer Fachoberschule": 12,
    "Realschulabschluss (Mittlere Reife)": 10,
    "Polytechnische Oberschule der DDR mit Abschluss der 10. Klasse": 10,
    "Hauptschulabschluss (Volksschulabschluss)": 9,
    "Polytechnische Oberschule der DDR mit Abschluss der 8. oder 9. Klasse": 9,
    "von der Schule abgegangen ohne Hauptschulabschluss (Volksschulabschluss)": 7,
    "Einen anderen Schulabschluss": 10,
    "Schüler:in, besuche eine allgemeinbildende Schule (Vollzeit oder Teilzeit)": np.nan,
    "Weiß nicht": np.nan,
    "Keine Angabe": np.nan,
}

vocational_years_map = {
    "Betriebliche Berufsausbildung (Lehre) abgeschlossen": 1.5,
    "Schulische Berufsausbildung abgeschlossen (Berufsfachschule, Handelsschule etc.)": 2.0,
    "Ausbildung an einer Fach-, Meister-, Technikerschule, Berufs- oder Fachakademie abgeschlossen": 3,
    "Ausbildung an einer Fachschule der DDR abgeschlossen": 3,
    "Bachelor an einer (Fach-) Hochschule abgeschlossen": 5,
    "Fachhochschulabschluss (z.B. Diplom, Master)": 5,
    "Universitätsabschluss (z.B. Diplom, Magister, Staatsexamen, Master)": 5,
    "Promotion": 8,
    "Keinen beruflichen Abschluss und nicht in beruflicher Ausbildung": 0,
    "Noch in beruflicher Ausbildung (Berufsvorbereitungsjahr, Ausbildung, Praktikum, Studium)": 0,
    "Schüler/in und besuche eine berufsorientierte Aufbau-, Fachschule o.ä.": 0,
    "Einen anderen beruflichen Abschluss": np.nan,
    "Weiß nicht": np.nan,
    "Keine Angabe": np.nan,
}

df['edu_school_years'] = df[edu_school_col].map(school_years_map)
voc_str = df[edu_coded_col].astype(str).str.strip()
df['edu_vocational_years'] = voc_str.map(vocational_years_map)

df['edu_total_years'] = np.nan
mask_add = df['edu_school_years'].notna() & (df['edu_school_years'] >= 7)
df.loc[mask_add, 'edu_total_years'] = (
    df.loc[mask_add, 'edu_school_years'] + df.loc[mask_add, 'edu_vocational_years']
)

# Pull education flags onto base index
miss_school = is_blank(df_base[school_col])
miss_edu_coded = is_blank(df_base[edu_coded_col])
miss_emp_pattern = ~employment_valid_ja_nein(df_base[employed_col])

edu_total = df.loc[df_base.index, 'edu_total_years']
unmappable_edu = edu_total.isna()

# “Unmappable” among people who did answer both education fields (substantive missing vs map failure)
both_edu_filled = ~miss_school & ~miss_edu_coded
unmappable_given_edu_filled = both_edu_filled & unmappable_edu

miss_phq = {c: miss_phq4_item(df_base[c]) for c in phq4_cols}
miss_behandlung = miss_lime_letter(df_base[behandlung_col], {'a', 'b'})
miss_diagnose = miss_lime_letter(df_base[diagnose_col], {'a', 'b', 'c'})
miss_eq5d = miss_lime_letter(df_base[eq5d_col], {'a', 'b', 'c', 'd', 'e'})

# --- Sociodemographic completeness (within base) ---
complete_soc = ~miss_emp_pattern & ~miss_school & ~miss_edu_coded

# --- Full clinical + soc completeness ---
complete_all = complete_soc & ~miss_behandlung & ~miss_diagnose & ~miss_eq5d
for c in phq4_cols:
    complete_all &= ~miss_phq[c]

# --- Print ---
print(f"Base sample (answered AI + gender + age): N = {n_base}\n")

print("Missing / invalid within base (each column):")
rows = [
    ("employment (not Ja/Nein as in regression)", miss_emp_pattern),
    ("school degree (blank)", miss_school),
    ("education_level_coded (blank)", miss_edu_coded),
    ("edu_total_years unmappable (NaN)", unmappable_edu),
    ("edu unmappable | school+coded filled", unmappable_given_edu_filled),
    ("Behandlung_dich (not a/b)", miss_behandlung),
    ("Diagnose_selbst_dich (not a/b/c)", miss_diagnose),
    ("EQ5D5L_3 (not a–e)", miss_eq5d),
]
for label, m in rows:
    print(f"  {label:<42} {int(m.sum()):>6}  ok: {int((~m).sum()):>6}")

for c in phq4_cols:
    m = miss_phq[c]
    print(f"  {c + ' (not 1–4)':<42} {int(m.sum()):>6}  ok: {int((~m).sum()):>6}")

print()
print(f"Complete on soc (emp Ja/Nein + school + coded):     {int(complete_soc.sum())}  ({100*complete_soc.mean():.1f}%)")
print(f"Complete on soc + clinical list:                   {int(complete_all.sum())}  ({100*complete_all.mean():.1f}%)")

Base sample (answered AI + gender + age): N = 28910

Missing / invalid within base (each column):
  employment (not Ja/Nein as in regression)     356  ok:  28554
  school degree (blank)                         312  ok:  28598
  education_level_coded (blank)                 324  ok:  28586
  edu_total_years unmappable (NaN)             2693  ok:  26217
  edu unmappable | school+coded filled         2353  ok:  26557
  Behandlung_dich (not a/b)                   15993  ok:  12917
  Diagnose_selbst_dich (not a/b/c)            15965  ok:  12945
  EQ5D5L_3 (not a–e)                          15935  ok:  12975
  PHQ4_1 (not 1–4)                            15965  ok:  12945
  PHQ4_2 (not 1–4)                            15982  ok:  12928
  PHQ4_3 (not 1–4)                            16005  ok:  12905
  PHQ4_4 (not 1–4)                            15997  ok:  12913

Complete on soc (emp Ja/Nein + school + coded):     28507  (98.6%)
Complete on soc + clinical list:                   12663  (43.8%)


In [5]:
import numpy as np
import pandas as pd

ki_col = 'ki_nutzen_sie_ki_programme_wie_z_b_chatgpt_gemini_claud'
gender_col = 'gender_filled'
age_col = 'birth_year_filled'
employed_col = 'sozdem_sind_sie_zurzeit_erwerbstaetig'
school_col = 'sozdem_welchen_hoechsten_allgemeinbildenden_schulabschl'
edu_coded_col = 'education_level_coded'
edu_school_col = school_col

phq4_cols = ['PHQ4_1', 'PHQ4_2', 'PHQ4_3', 'PHQ4_4']
diagnose_col = 'Diagnose_selbst_dich'
eq5d_col = 'EQ5D5L_3'
behandlung_col = 'Behandlung_dich'


def is_blank(s: pd.Series) -> pd.Series:
    return s.isna() | (s.astype(str).str.strip() == '')


def miss_phq4_item(s: pd.Series) -> pd.Series:
    n = pd.to_numeric(s, errors='coerce')
    return ~n.isin([1, 2, 3, 4])


def miss_lime_letter(s: pd.Series, valid: set[str]) -> pd.Series:
    v = s.astype(str).str.strip().str.lower()
    blank = s.isna() | v.eq('') | v.isin(['nan', 'none', '<na>'])
    return blank | ~v.isin(valid)


# --- Base: answered AI + gender + age ---
answered_ai = ~is_blank(df[ki_col])
has_demo = answered_ai & ~is_blank(df[gender_col]) & ~is_blank(df[age_col])
df_base = df.loc[has_demo]
n_base = len(df_base)

miss_employed = is_blank(df_base[employed_col])
miss_school = is_blank(df_base[school_col])
miss_edu_coded = is_blank(df_base[edu_coded_col])

# Sociodemographics only (within base): employment + school + coded education
complete_soc = ~miss_employed & ~miss_school & ~miss_edu_coded
idx_soc = df_base.index[complete_soc]
n_soc = int(len(idx_soc))

# Full list: soc + clinical
miss_phq = {c: miss_phq4_item(df_base[c]) for c in phq4_cols}
miss_behandlung = miss_lime_letter(df_base[behandlung_col], {'a', 'b'})
miss_diagnose = miss_lime_letter(df_base[diagnose_col], {'a', 'b', 'c'})
miss_eq5d = miss_lime_letter(df_base[eq5d_col], {'a', 'b', 'c', 'd', 'e'})

complete_clinical = complete_soc.copy()
complete_clinical &= ~miss_behandlung & ~miss_diagnose & ~miss_eq5d
for c in phq4_cols:
    complete_clinical &= ~miss_phq[c]

idx_clinical = df_base.index[complete_clinical]
n_clinical = int(len(idx_clinical))

# --- Education → years (on full df; same as your notebook) ---
school_years_map = {
    "Allgemeine oder fachgebundene Hochschulreife/Abitur (Gymnasium bzw. Abschluss der Erweiterten Oberschule (der DDR))": 13,
    "Fachhochschulreife, Abschluss einer Fachoberschule": 12,
    "Realschulabschluss (Mittlere Reife)": 10,
    "Polytechnische Oberschule der DDR mit Abschluss der 10. Klasse": 10,
    "Hauptschulabschluss (Volksschulabschluss)": 9,
    "Polytechnische Oberschule der DDR mit Abschluss der 8. oder 9. Klasse": 9,
    "von der Schule abgegangen ohne Hauptschulabschluss (Volksschulabschluss)": 7,
    "Einen anderen Schulabschluss": 10,
    "Schüler:in, besuche eine allgemeinbildende Schule (Vollzeit oder Teilzeit)": np.nan,
    "Weiß nicht": np.nan,
    "Keine Angabe": np.nan,
}

vocational_years_map = {
    "Betriebliche Berufsausbildung (Lehre) abgeschlossen": 1.5,
    "Schulische Berufsausbildung abgeschlossen (Berufsfachschule, Handelsschule etc.)": 2.0,
    "Ausbildung an einer Fach-, Meister-, Technikerschule, Berufs- oder Fachakademie abgeschlossen": 3,
    "Ausbildung an einer Fachschule der DDR abgeschlossen": 3,
    "Bachelor an einer (Fach-) Hochschule abgeschlossen": 5,
    "Fachhochschulabschluss (z.B. Diplom, Master)": 5,
    "Universitätsabschluss (z.B. Diplom, Magister, Staatsexamen, Master)": 5,
    "Promotion": 8,
    "Keinen beruflichen Abschluss und nicht in beruflicher Ausbildung": 0,
    "Noch in beruflicher Ausbildung (Berufsvorbereitungsjahr, Ausbildung, Praktikum, Studium)": 0,
    "Schüler/in und besuche eine berufsorientierte Aufbau-, Fachschule o.ä.": 0,
    "Einen anderen beruflichen Abschluss": np.nan,
    "Weiß nicht": np.nan,
    "Keine Angabe": np.nan,
}

df['edu_school_years'] = df[edu_school_col].map(school_years_map)
voc_str = df[edu_coded_col].astype(str).str.strip()
df['edu_vocational_years'] = voc_str.map(vocational_years_map)

df['edu_total_years'] = np.nan
mask_add = df['edu_school_years'].notna() & (df['edu_school_years'] >= 7)
df.loc[mask_add, 'edu_total_years'] = (
    df.loc[mask_add, 'edu_school_years'] + df.loc[mask_add, 'edu_vocational_years']
)


def report_edu_mapping(label: str, idx: pd.Index) -> None:
    sub = df.loc[idx]
    mapped = sub['edu_total_years'].notna()
    n = len(sub)
    n_ok = int(mapped.sum())
    n_na = n - n_ok
    print(f"{label}")
    print(f"  N in group:                        {n}")
    print(f"  N with mappable edu_total_years:   {n_ok} ({100 * n_ok / n:.1f}%)")
    print(f"  N without edu_total_years (NaN):     {n_na} ({100 * n_na / n:.1f}%)")
    print()


print(f"Base sample (answered AI + gender + age): N = {n_base}\n")

report_edu_mapping(
    "A) Sociodemographics only (employment + school + education_level_coded, within base):",
    idx_soc,
)
report_edu_mapping(
    "B) Sociodemographics + all clinical fields (PHQ4_1–4, Behandlung_dich, Diagnose_selbst_dich, EQ5D5L_3), within base:",
    idx_clinical,
)

Base sample (answered AI + gender + age): N = 28910

A) Sociodemographics only (employment + school + education_level_coded, within base):
  N in group:                        28507
  N with mappable edu_total_years:   26162 (91.8%)
  N without edu_total_years (NaN):     2345 (8.2%)

B) Sociodemographics + all clinical fields (PHQ4_1–4, Behandlung_dich, Diagnose_selbst_dich, EQ5D5L_3), within base:
  N in group:                        12663
  N with mappable edu_total_years:   11628 (91.8%)
  N without edu_total_years (NaN):     1035 (8.2%)



## Missingness assessment: sociodemographics and clinical variables

Here we extend the missingness check to include both sociodemographic variables and clinical variables.


In [6]:
import numpy as np
import pandas as pd

ki_col = 'ki_nutzen_sie_ki_programme_wie_z_b_chatgpt_gemini_claud'
gender_col = 'gender_filled'
age_col = 'birth_year_filled'
employed_col = 'sozdem_sind_sie_zurzeit_erwerbstaetig'
school_col = 'sozdem_welchen_hoechsten_allgemeinbildenden_schulabschl'
edu_coded_col = 'education_level_coded'

phq4_cols = ['PHQ4_1', 'PHQ4_2', 'PHQ4_3', 'PHQ4_4']
diagnose_col = 'Diagnose_selbst_dich'
eq5d_col = 'EQ5D5L_3'
behandlung_col = 'Behandlung_dich'


def is_blank(s: pd.Series) -> pd.Series:
    return s.isna() | (s.astype(str).str.strip() == '')


def miss_phq4_item(s: pd.Series) -> pd.Series:
    n = pd.to_numeric(s, errors='coerce')
    return ~n.isin([1, 2, 3, 4])


def miss_lime_letter(s: pd.Series, valid: set[str]) -> pd.Series:
    v = s.astype(str).str.strip().str.lower()
    blank = s.isna() | v.eq('') | v.isin(['nan', 'none', '<na>'])
    return blank | ~v.isin(valid)


# Who answered AI (reference only)
answered_ai = ~is_blank(df[ki_col])
n_answered_ai = int(answered_ai.sum())

# Base sample: answered AI + non-missing gender + non-missing age  → your ~28910
has_demo = answered_ai & ~is_blank(df[gender_col]) & ~is_blank(df[age_col])
df_base = df.loc[has_demo].copy()
n_base = len(df_base)

# Per-variable missing within THIS base (gender/age are complete by construction)
miss_gender = is_blank(df_base[gender_col])
miss_age = is_blank(df_base[age_col])
miss_employed = is_blank(df_base[employed_col])
miss_school = is_blank(df_base[school_col])
miss_edu_coded = is_blank(df_base[edu_coded_col])

miss_phq = {c: miss_phq4_item(df_base[c]) for c in phq4_cols}

miss_behandlung = miss_lime_letter(df_base[behandlung_col], {'a', 'b'})
miss_diagnose = miss_lime_letter(df_base[diagnose_col], {'a', 'b', 'c'})
miss_eq5d = miss_lime_letter(df_base[eq5d_col], {'a', 'b', 'c', 'd', 'e'})

complete_all = (
    ~miss_gender
    & ~miss_age
    & ~miss_employed
    & ~miss_school
    & ~miss_edu_coded
    & ~miss_behandlung
    & ~miss_diagnose
    & ~miss_eq5d
)
for c in phq4_cols:
    complete_all &= ~miss_phq[c]

n_complete = int(complete_all.sum())
n_incomplete = int((~complete_all).sum())

print("Reference: answered AI question (any gender/age):")
print(f"  N: {n_answered_ai}")
print()
print("Base sample: answered AI + non-missing gender + non-missing age:")
print(f"  N: {n_base}")
print()
print("Within that base sample:")
print(
    "  N complete on employment, school, education_level_coded, "
    "PHQ4_1–4, Diagnose_selbst_dich, EQ5D5L_3, Behandlung_dich:"
)
print(f"    {n_complete} ({100 * n_complete / n_base:.1f}%)")
print(f"  N missing ≥1 of those: {n_incomplete} ({100 * n_incomplete / n_base:.1f}%)")
print()

print("Missing counts within base sample (each column separately):")
rows = [
    ("gender_filled", gender_col, miss_gender),
    ("birth_year_filled", age_col, miss_age),
    ("employment", employed_col, miss_employed),
    ("school degree", school_col, miss_school),
    ("education_level_coded", edu_coded_col, miss_edu_coded),
    ("Behandlung_dich", behandlung_col, miss_behandlung),
    ("Diagnose_selbst_dich", diagnose_col, miss_diagnose),
    ("EQ5D5L_3", eq5d_col, miss_eq5d),
]
for name, _col, m in rows:
    print(f"  {name:<28} missing: {int(m.sum()):>6}  non-missing: {int((~m).sum()):>6}")

for c in phq4_cols:
    m = miss_phq[c]
    print(f"  {c:<28} missing: {int(m.sum()):>6}  non-missing: {int((~m).sum()):>6}")

Reference: answered AI question (any gender/age):
  N: 29045

Base sample: answered AI + non-missing gender + non-missing age:
  N: 28910

Within that base sample:
  N complete on employment, school, education_level_coded, PHQ4_1–4, Diagnose_selbst_dich, EQ5D5L_3, Behandlung_dich:
    12663 (43.8%)
  N missing ≥1 of those: 16247 (56.2%)

Missing counts within base sample (each column separately):
  gender_filled                missing:      0  non-missing:  28910
  birth_year_filled            missing:      0  non-missing:  28910
  employment                   missing:    356  non-missing:  28554
  school degree                missing:    312  non-missing:  28598
  education_level_coded        missing:    324  non-missing:  28586
  Behandlung_dich              missing:  15993  non-missing:  12917
  Diagnose_selbst_dich         missing:  15965  non-missing:  12945
  EQ5D5L_3                     missing:  15935  non-missing:  12975
  PHQ4_1                       missing:  15965  non-miss

In [7]:
import numpy as np
import pandas as pd

# --- Reuse same column names and helpers as your complete-case cell ---
ki_col = 'ki_nutzen_sie_ki_programme_wie_z_b_chatgpt_gemini_claud'
gender_col = 'gender_filled'
age_col = 'birth_year_filled'
employed_col = 'sozdem_sind_sie_zurzeit_erwerbstaetig'
school_col = 'sozdem_welchen_hoechsten_allgemeinbildenden_schulabschl'
edu_coded_col = 'education_level_coded'
phq4_cols = ['PHQ4_1', 'PHQ4_2', 'PHQ4_3', 'PHQ4_4']
diagnose_col = 'Diagnose_selbst_dich'
eq5d_col = 'EQ5D5L_3'
behandlung_col = 'Behandlung_dich'

edu_school_col = school_col


def is_blank(s: pd.Series) -> pd.Series:
    return s.isna() | (s.astype(str).str.strip() == '')


def miss_phq4_item(s: pd.Series) -> pd.Series:
    n = pd.to_numeric(s, errors='coerce')
    return ~n.isin([1, 2, 3, 4])


def miss_lime_letter(s: pd.Series, valid: set[str]) -> pd.Series:
    v = s.astype(str).str.strip().str.lower()
    blank = s.isna() | v.eq('') | v.isin(['nan', 'none', '<na>'])
    return blank | ~v.isin(valid)


# --- Base: answered AI + gender + age ---
answered_ai = ~is_blank(df[ki_col])
has_demo = answered_ai & ~is_blank(df[gender_col]) & ~is_blank(df[age_col])
df_base = df.loc[has_demo]

miss_employed = is_blank(df_base[employed_col])
miss_school = is_blank(df_base[school_col])
miss_edu_coded = is_blank(df_base[edu_coded_col])
miss_phq = {c: miss_phq4_item(df_base[c]) for c in phq4_cols}
miss_behandlung = miss_lime_letter(df_base[behandlung_col], {'a', 'b'})
miss_diagnose = miss_lime_letter(df_base[diagnose_col], {'a', 'b', 'c'})
miss_eq5d = miss_lime_letter(df_base[eq5d_col], {'a', 'b', 'c', 'd', 'e'})

complete_clinical = (
    ~miss_employed
    & ~miss_school
    & ~miss_edu_coded
    & ~miss_behandlung
    & ~miss_diagnose
    & ~miss_eq5d
)
for c in phq4_cols:
    complete_clinical &= ~miss_phq[c]

idx_complete = df_base.index[complete_clinical]
n_complete_clinical = int(len(idx_complete))

# --- Your education mappings (full df so columns align with index) ---
school_years_map = {
    "Allgemeine oder fachgebundene Hochschulreife/Abitur (Gymnasium bzw. Abschluss der Erweiterten Oberschule (der DDR))": 13,
    "Fachhochschulreife, Abschluss einer Fachoberschule": 12,
    "Realschulabschluss (Mittlere Reife)": 10,
    "Polytechnische Oberschule der DDR mit Abschluss der 10. Klasse": 10,
    "Hauptschulabschluss (Volksschulabschluss)": 9,
    "Polytechnische Oberschule der DDR mit Abschluss der 8. oder 9. Klasse": 9,
    "von der Schule abgegangen ohne Hauptschulabschluss (Volksschulabschluss)": 7,
    "Einen anderen Schulabschluss": 10,
    "Schüler:in, besuche eine allgemeinbildende Schule (Vollzeit oder Teilzeit)": np.nan,
    "Weiß nicht": np.nan,
    "Keine Angabe": np.nan,
}

vocational_years_map = {
    "Betriebliche Berufsausbildung (Lehre) abgeschlossen": 1.5,
    "Schulische Berufsausbildung abgeschlossen (Berufsfachschule, Handelsschule etc.)": 2.0,
    "Ausbildung an einer Fach-, Meister-, Technikerschule, Berufs- oder Fachakademie abgeschlossen": 3,
    "Ausbildung an einer Fachschule der DDR abgeschlossen": 3,
    "Bachelor an einer (Fach-) Hochschule abgeschlossen": 5,
    "Fachhochschulabschluss (z.B. Diplom, Master)": 5,
    "Universitätsabschluss (z.B. Diplom, Magister, Staatsexamen, Master)": 5,
    "Promotion": 8,
    "Keinen beruflichen Abschluss und nicht in beruflicher Ausbildung": 0,
    "Noch in beruflicher Ausbildung (Berufsvorbereitungsjahr, Ausbildung, Praktikum, Studium)": 0,
    "Schüler/in und besuche eine berufsorientierte Aufbau-, Fachschule o.ä.": 0,
    "Einen anderen beruflichen Abschluss": np.nan,
    "Weiß nicht": np.nan,
    "Keine Angabe": np.nan,
}

df['edu_school_years'] = df[edu_school_col].map(school_years_map)
voc_str = df[edu_coded_col].astype(str).str.strip()
df['edu_vocational_years'] = voc_str.map(vocational_years_map)

df['edu_total_years'] = np.nan
mask_add = df['edu_school_years'].notna() & (df['edu_school_years'] >= 7)
df.loc[mask_add, 'edu_total_years'] = (
    df.loc[mask_add, 'edu_school_years'] + df.loc[mask_add, 'edu_vocational_years']
)

# --- Among clinically complete: how many get a defined edu_total_years? ---
sub = df.loc[idx_complete]
mapped = sub['edu_total_years'].notna()
n_mapped = int(mapped.sum())
n_not_mapped = int((~mapped).sum())

print("Among participants with answered AI + gender + age + all clinical/soc fields:")
print(f"  N (complete on clinical list):     {n_complete_clinical}")
print(f"  N with mappable edu_total_years:   {n_mapped} ({100 * n_mapped / n_complete_clinical:.1f}%)")
print(f"  N without edu_total_years (NaN):     {n_not_mapped} ({100 * n_not_mapped / n_complete_clinical:.1f}%)")

Among participants with answered AI + gender + age + all clinical/soc fields:
  N (complete on clinical list):     12663
  N with mappable edu_total_years:   11628 (91.8%)
  N without edu_total_years (NaN):     1035 (8.2%)


In [8]:
# -*- coding: utf-8 -*-
# =============================================================================
# Analysis sample exclusions: missingness in age, gender, and AI-use response
# =============================================================================
# Purpose: Quantify exclusions due to missing key variables and report how often
#          age/gender missingness occurs among participants who did vs. did not
#          provide an AI-use response.
# Input:   Uses `df` loaded earlier in the notebook.
# Output:  Printed exclusion counts and missingness breakdowns.
# =============================================================================


# Column names
ki_col = 'ki_nutzen_sie_ki_programme_wie_z_b_chatgpt_gemini_claud'
gender_col = 'gender_filled'
age_col = 'birth_year_filled'

total_n = len(df)

# 1. Excluded due to missing AI usage answer
missing_ki = df[ki_col].isna() | (df[ki_col].astype(str).str.strip() == '')
n_missing_ki = missing_ki.sum()

# 2. Excluded due to missing gender and/or age
missing_gender = df[gender_col].isna() | (df[gender_col].astype(str).str.strip() == '')
missing_age = df[age_col].isna() | (df[age_col].astype(str).str.strip() == '')
missing_gender_or_age = missing_gender | missing_age
n_missing_gender_or_age = missing_gender_or_age.sum()

# 3. Excluded due to missing ONLY AI usage (provided gender and age)
missing_ki_and_not_missing_gender_or_age = missing_ki & (~missing_gender_or_age)
n_missing_only_ki = missing_ki_and_not_missing_gender_or_age.sum()

# 4. Excluded due to missing ONLY gender/age (provided AI usage)
missing_gender_or_age_and_not_missing_ki = (~missing_ki) & missing_gender_or_age
n_missing_only_gender_or_age = missing_gender_or_age_and_not_missing_ki.sum()

# 5. Excluded due to BOTH missing AI usage AND missing gender/age
missing_both_ki_and_gender_or_age = missing_ki & missing_gender_or_age
n_missing_both = missing_both_ki_and_gender_or_age.sum()

# Calculate exclusion counts as before (full details later in cell), but split gender/age
# missingness by whether the participant did/didn't provide an AI usage response.

# Mask for missing
missing_ki = df[ki_col].isna() | (df[ki_col].astype(str).str.strip() == '')
missing_gender = df[gender_col].isna() | (df[gender_col].astype(str).str.strip() == '')
missing_age = df[age_col].isna() | (df[age_col].astype(str).str.strip() == '')

# Of those missing gender, how many gave an AI usage answer?
n_missing_gender = missing_gender.sum()
n_missing_gender_ai_answer = (missing_gender & ~missing_ki).sum()

# Of those missing age, how many gave an AI usage answer?
n_missing_age = missing_age.sum()
n_missing_age_ai_answer = (missing_age & ~missing_ki).sum()

# Print breakdowns
print(f'Total missing gender: {n_missing_gender}')
print(f'   — of which provided AI usage answer: {n_missing_gender_ai_answer}')
print(f'Total missing age: {n_missing_age}')
print(f'   — of which provided AI usage answer: {n_missing_age_ai_answer}')

# Print results
print(f'Total participants: {total_n}')
print(f'Excluded due to missing AI usage answer: {n_missing_ki}')
print(f'   — of which provided gender AND age: {n_missing_only_ki}')
print(f'Excluded due to missing gender and/or age: {n_missing_gender_or_age}')
print(f'   — of which provided AI usage answer: {n_missing_only_gender_or_age}')
print(f'Excluded due to BOTH missing AI usage AND missing gender/age: {n_missing_both}')

Total missing gender: 100
   — of which provided AI usage answer: 99
Total missing age: 127
   — of which provided AI usage answer: 125
Total participants: 29571
Excluded due to missing AI usage answer: 526
   — of which provided gender AND age: 524
Excluded due to missing gender and/or age: 137
   — of which provided AI usage answer: 135
Excluded due to BOTH missing AI usage AND missing gender/age: 2


## Sample comparison: clinical questionnaire shown vs. not shown

We compare participants who were **presented** the clinical questionnaire to those who were **not presented** it, to assess  differences between these subsamples.


In [9]:
import numpy as np
import pandas as pd
from scipy import stats

# =============================================================================
# 1) LOAD DATA
# =============================================================================
# df = pd.read_csv(r"C:\path\to\your_file.csv", sep=";", low_memory=False, on_bad_lines="skip")

# =============================================================================
# 2) COLUMN NAMES
# =============================================================================
ki_col = "ki_nutzen_sie_ki_programme_wie_z_b_chatgpt_gemini_claud"
gender_col = "gender_filled"
age_col = "birth_year_filled"
employed_col = "sozdem_sind_sie_zurzeit_erwerbstaetig"
school_col = "sozdem_welchen_hoechsten_allgemeinbildenden_schulabschl"
edu_coded_col = "education_level_coded"
edu_school_col = school_col
COL_PAGE = "lastpage_mds"
REFERENCE_YEAR = 2026

purpose_cols_1_3 = [
    "ki_wie_haeufig_nutzen_sie_ki_programme_fuer_die_folgend",
    "ki_wie_haeufig_nutzen_sie_ki_programme_fuer_die_folgend_2",
    "ki_wie_haeufig_nutzen_sie_ki_programme_fuer_die_folgend_3",
]
purpose_cols_4_13 = [
    "ki_bei_aengsten_und_sorgen",
    "ki_bei_selbstzweifeln_oder_niedergeschlagenheit",
    "ki_bei_einsamkeit_z_b_um_jemanden_zum_reden_zu_haben_we",
    "ki_bei_zwischenmenschlichen_konflikten",
    "ki_bei_unsicherheiten_bei_entscheidungen_um_meine_gedan",
    "ki_bei_trauer_oder_verlust",
    "ki_wenn_grosse_lebensveraenderungen_anstehen",
    "ki_wie_haeufig_nutzen_sie_ki_programme_fuer_die_folgend_4",
    "ki_wie_haeufig_nutzen_sie_ki_programme_fuer_die_folgend_5",
    "ki_fuer_sonstige_gesundheits_psychische_oder_emotionale",
]
col_grief = "ki_bei_trauer_oder_verlust"
col_life = "ki_wenn_grosse_lebensveraenderungen_anstehen"

side_effect_cols = [
    "ki_mein_verhaeltnis_zu_meinen_freunden_ist_seit_meiner",
    "ki_ai_dependency",
    "ki_decision_difficulty",
]
SOCIAL_YES_NUM = {-1, -2, -3}
YES_VALUES_TEXT = {"trifft ein wenig zu", "trifft teilweise zu", "trifft völlig zu", "trifft voellig zu"}


def ensure_shown_clinical(frame: pd.DataFrame, page_col: str = COL_PAGE) -> pd.DataFrame:
    if "shown_clinical" in frame.columns:
        return frame
    if page_col not in frame.columns:
        raise KeyError(
            f"Need {page_col!r} to build shown_clinical. "
            f"First columns: {list(frame.columns)[:30]}"
        )
    out = frame.copy()
    lp = out[page_col]
    shown = lp.notna()
    if lp.dtype == object or pd.api.types.is_string_dtype(lp):
        strv = lp.astype(str).str.strip()
        shown = shown & (strv != "") & (strv.str.lower() != "nan")
    out["shown_clinical"] = shown.astype(int)
    return out


def classify_ki_answer(v):
    if pd.isna(v):
        return "missing"
    s = str(v).strip()
    if s == "" or s.lower() == "nan":
        return "missing"
    low = s.lower()
    if low.startswith("ja") or low in ("1", "y", "yes", "true"):
        return "yes"
    if low.startswith("nein") or low in ("0", "2", "n", "no", "false"):
        return "no"
    return "other"


df = df.copy()
df = ensure_shown_clinical(df)

print("=== Unique values: KI use question (first 30 by frequency) ===")
if ki_col not in df.columns:
    raise KeyError(f"Missing column {ki_col!r}")
vc = df[ki_col].astype(str).str.strip().replace({"nan": np.nan}).value_counts(dropna=False)
print(vc.head(30))

df["ki_answer_class"] = df[ki_col].apply(classify_ki_answer)

# Age (needed for analysis sample filter)
df["age_2026"] = REFERENCE_YEAR - pd.to_numeric(df[age_col], errors="coerce")

# =============================================================================
# ANALYSIS SAMPLE: KI Ja/Nein + non-missing gender + non-missing age
# =============================================================================
n0 = len(df)
miss_ki = ~df["ki_answer_class"].isin(["yes", "no"])
gstr = df[gender_col].astype(str).str.strip()
miss_gender = df[gender_col].isna() | gstr.eq("") | gstr.str.lower().eq("nan")
miss_age = df["age_2026"].isna()
mask_keep = (~miss_ki) & (~miss_gender) & (~miss_age)

print("\n=== Analysis sample restriction (KI yes/no + gender + age) ===")
print(f"  N total:                    {n0:,}")
print(f"  Excl. not Ja/Nein on KI:    {int(miss_ki.sum()):,}")
print(f"  Excl. missing gender:       {int(miss_gender.sum()):,}")
print(f"  Excl. missing age:          {int(miss_age.sum()):,}")
print(f"  (Reasons overlap; kept:)    {int(mask_keep.sum()):,}")

df = df.loc[mask_keep].reset_index(drop=True)
print(f"  Final analysis N:           {len(df):,}")

# =============================================================================
# Sociodemographics + edu_total_years (on analysis sample only)
# =============================================================================
g = df[gender_col].astype(str).str.strip().str.lower()
male = np.full(len(df), np.nan)
male[g.str.contains(r"männ|maenn|männlich|^m$", na=False, regex=True)] = 1.0
male[g.str.contains(r"weib|weiblich|^w$|^f$", na=False, regex=True)] = 0.0
df["male"] = male

df["gender_cat"] = np.where(
    df["male"] == 1,
    "Male",
    np.where(df["male"] == 0, "Female", "Other"),
)

s_emp = df[employed_col].astype(str).str.strip().str.lower()
df["employed"] = np.where(
    s_emp.str.startswith("ja"), 1.0, np.where(s_emp.str.startswith("nein"), 0.0, np.nan)
)

school_years_map = {
    "Allgemeine oder fachgebundene Hochschulreife/Abitur (Gymnasium bzw. Abschluss der Erweiterten Oberschule (der DDR))": 13,
    "Fachhochschulreife, Abschluss einer Fachoberschule": 12,
    "Realschulabschluss (Mittlere Reife)": 10,
    "Polytechnische Oberschule der DDR mit Abschluss der 10. Klasse": 10,
    "Hauptschulabschluss (Volksschulabschluss)": 9,
    "Polytechnische Oberschule der DDR mit Abschluss der 8. oder 9. Klasse": 9,
    "von der Schule abgegangen ohne Hauptschulabschluss (Volksschulabschluss)": 7,
    "Einen anderen Schulabschluss": 10,
    "Schüler:in, besuche eine allgemeinbildende Schule (Vollzeit oder Teilzeit)": np.nan,
    "Weiß nicht": np.nan,
    "Keine Angabe": np.nan,
}
vocational_years_map = {
    "Betriebliche Berufsausbildung (Lehre) abgeschlossen": 1.5,
    "Schulische Berufsausbildung abgeschlossen (Berufsfachschule, Handelsschule etc.)": 2.0,
    "Ausbildung an einer Fach-, Meister-, Technikerschule, Berufs- oder Fachakademie abgeschlossen": 3,
    "Ausbildung an einer Fachschule der DDR abgeschlossen": 3,
    "Bachelor an einer (Fach-) Hochschule abgeschlossen": 5,
    "Fachhochschulabschluss (z.B. Diplom, Master)": 5,
    "Universitätsabschluss (z.B. Diplom, Magister, Staatsexamen, Master)": 5,
    "Promotion": 8,
    "Keinen beruflichen Abschluss und nicht in beruflicher Ausbildung": 0,
    "Noch in beruflicher Ausbildung (Berufsvorbereitungsjahr, Ausbildung, Praktikum, Studium)": 0,
    "Schüler/in und besuche eine berufsorientierte Aufbau-, Fachschule o.ä.": 0,
    "Einen anderen beruflichen Abschluss": np.nan,
    "Weiß nicht": np.nan,
    "Keine Angabe": np.nan,
}

df["edu_school_years"] = df[edu_school_col].map(school_years_map)
voc_str = df[edu_coded_col].astype(str).str.strip()
df["edu_vocational_years"] = voc_str.map(vocational_years_map)
df["edu_total_years"] = np.nan
mask_add = df["edu_school_years"].notna() & (df["edu_school_years"] >= 7)
df.loc[mask_add, "edu_total_years"] = (
    df.loc[mask_add, "edu_school_years"] + df.loc[mask_add, "edu_vocational_years"]
)

# =============================================================================
# Q4 intensity + flags
# =============================================================================
def norm_txt(x):
    if pd.isna(x):
        return ""
    s = str(x).strip().lower()
    if s == "nan":
        return ""
    return s.replace("ä", "ae").replace("ö", "oe").replace("ü", "ue").replace("ß", "ss")


intensity_map = {"selten": 1, "manchmal": 2, "oft": 3, "immer": 4}
non_use_answers = {
    "hierfuer nutze ich keine ki",
    "(bisher) kein bedarf, wuerde ki aber dafuer nutzen",
}


def map_intensity(v):
    s = norm_txt(v)
    if s == "":
        return 0
    if s in non_use_answers:
        return 0
    if s in intensity_map:
        return intensity_map[s]
    for k in intensity_map:
        if k in s:
            return intensity_map[k]
    return np.nan


for cols in [purpose_cols_1_3, purpose_cols_4_13]:
    for c in cols:
        if c in df.columns:
            df[f"{c}_int"] = df[c].apply(map_intensity)

if col_grief in df.columns and col_life in df.columns:
    df["grief_life_combined_int"] = df[
        [f"{col_grief}_int", f"{col_life}_int"]
    ].max(axis=1)
else:
    df["grief_life_combined_int"] = np.nan

int_13 = [f"{c}_int" for c in purpose_cols_1_3 if c in df.columns]
all_413 = [f"{c}_int" for c in purpose_cols_4_13 if c in df.columns]
int_413 = [
    c for c in all_413 if c not in [f"{col_grief}_int", f"{col_life}_int"]
] + (["grief_life_combined_int"] if "grief_life_combined_int" in df.columns else [])


def any_substantive_row(row, int_cols):
    if not int_cols:
        return np.nan
    vals = [row[c] for c in int_cols if c in row.index]
    return any(pd.notna(v) and v > 0 for v in vals)


df["any_use_q4_1_3"] = df.apply(lambda r: any_substantive_row(r, int_13), axis=1)
df["any_use_q4_4_13"] = df.apply(lambda r: any_substantive_row(r, int_413), axis=1)


def to_yes_no(v, col):
    if pd.isna(v):
        return np.nan
    s = str(v).strip()
    if s == "" or s.lower() == "nan":
        return np.nan
    if col == side_effect_cols[0]:
        num = pd.to_numeric(s, errors="coerce")
        if pd.notna(num) and int(num) in SOCIAL_YES_NUM:
            return 1
        return 0
    return 1 if s.lower() in YES_VALUES_TEXT else 0


ycols = []
for col in side_effect_cols:
    if col in df.columns:
        df[f"{col}_yes"] = df[col].apply(lambda x, c=col: to_yes_no(x, c))
        ycols.append(f"{col}_yes")

if ycols:
    answered_any = df[ycols].notna().any(axis=1)
    any_yes = df[ycols].eq(1).any(axis=1)
    df["side_effect_any"] = np.where(~answered_any, np.nan, np.where(any_yes, 1, 0))
else:
    df["side_effect_any"] = np.nan


# =============================================================================
# KI × shown_clinical
# =============================================================================
print("\n" + "=" * 70)
print("KI USE QUESTION × shown_clinical (raw counts) — analysis sample only")
print("=" * 70)

ct_ki = pd.crosstab(df["shown_clinical"], df["ki_answer_class"], margins=True)
print(ct_ki)

print("\n--- % answering YES among those with class yes/no (excluding missing) ---")
for sc, lab in [(0, "Not shown clinical"), (1, "Shown clinical")]:
    sub = df[df["shown_clinical"] == sc]
    denom = sub["ki_answer_class"].isin(["yes", "no"]).sum()
    n_yes = (sub["ki_answer_class"] == "yes").sum()
    if denom:
        print(f"{lab}: Yes = {n_yes} / {denom} = {100 * n_yes / denom:.1f}%")
    else:
        print(f"{lab}: no yes/no answers")

print("\n--- % YES among ALL rows in group ---")
for sc, lab in [(0, "Not shown clinical"), (1, "Shown clinical")]:
    sub = df[df["shown_clinical"] == sc]
    n_all = len(sub)
    n_yes = (sub["ki_answer_class"] == "yes").sum()
    print(f"{lab}: Yes = {n_yes} / {n_all} = {100 * n_yes / n_all:.1f}%")

ct2 = pd.crosstab(df["shown_clinical"], df["ki_answer_class"] == "yes")
if ct2.shape == (2, 2):
    print("Chi-square (shown × binary Yes vs not Yes): p =", stats.chi2_contingency(ct2)[1])

df["ki_yes_binary"] = (df["ki_answer_class"] == "yes").astype(float)
df.loc[df["ki_answer_class"] == "missing", "ki_yes_binary"] = np.nan

ct_yes = pd.crosstab(df["shown_clinical"], df["ki_yes_binary"])
print("\n2×2 shown_clinical × ki_yes_binary (1=yes, 0=no/other; missing excluded):")
print(ct_yes)
sub = df.dropna(subset=["ki_yes_binary", "shown_clinical"])
if pd.crosstab(sub["shown_clinical"], sub["ki_yes_binary"]).shape == (2, 2):
    print("Chi-square p:", stats.chi2_contingency(pd.crosstab(sub["shown_clinical"], sub["ki_yes_binary"]))[1])


# =============================================================================
# Q4 — raw counts
# =============================================================================
print("\n" + "=" * 70)
print("Q4 substantive use — raw counts — analysis sample only")
print("=" * 70)

for colbool, title in [
    ("any_use_q4_1_3", "Any substantive use Q4.1–Q4.3"),
    ("any_use_q4_4_13", "Any substantive use Q4.4–Q4.13"),
]:
    print(f"\n--- {title} ---")
    ct = pd.crosstab(df["shown_clinical"], df[colbool], margins=True)
    print(ct)
    sub = df.dropna(subset=[colbool, "shown_clinical"])
    t = sub[sub[colbool].isin([True, False])]
    ct2 = pd.crosstab(t["shown_clinical"], t[colbool])
    print("Among non-missing boolean:")
    print(ct2)
    if ct2.shape == (2, 2):
        print("Chi-square p:", stats.chi2_contingency(ct2)[1])


# =============================================================================
# Side effects
# =============================================================================
print("\n" + "=" * 70)
print("Side effects — raw counts (≥1 side-effect item answered) — analysis sample only")
print("=" * 70)
sub = df.dropna(subset=["side_effect_any", "shown_clinical"])
ct = pd.crosstab(sub["shown_clinical"], sub["side_effect_any"], margins=True)
print(ct)
ct2 = pd.crosstab(sub["shown_clinical"], sub["side_effect_any"])
if ct2.shape == (2, 2):
    print("Chi-square p:", stats.chi2_contingency(ct2)[1])


# =============================================================================
# Sociodemographics
# =============================================================================
print("\n" + "=" * 70)
print("Sociodemographics — analysis sample only")
print("=" * 70)

g0 = df[df["shown_clinical"] == 0]
g1 = df[df["shown_clinical"] == 1]
print(f"Not shown clinical: n = {len(g0):,}")
print(f"Shown clinical:     n = {len(g1):,}")

a = df.dropna(subset=["age_2026", "shown_clinical"]).copy()
a0, a1 = a.loc[a["shown_clinical"] == 0, "age_2026"], a.loc[a["shown_clinical"] == 1, "age_2026"]
print("\n--- Age (years) ---")
print(f"Not shown: mean {a0.mean():.2f}, SD {a0.std():.2f}, median {a0.median():.1f}, n={len(a0)}")
print(f"Shown:     mean {a1.mean():.2f}, SD {a1.std():.2f}, median {a1.median():.1f}, n={len(a1)}")
print("Welch t-test p:", stats.ttest_ind(a1, a0, equal_var=False).pvalue)
print("Mann–Whitney p:", stats.mannwhitneyu(a1, a0, alternative="two-sided").pvalue)

sx = df.dropna(subset=["gender_cat", "shown_clinical"]).copy()
ct_g = pd.crosstab(sx["shown_clinical"], sx["gender_cat"])
print("\n--- Sex (Male / Female / Other) — counts ---")
print(ct_g)
print("\n--- Sex — % within shown_clinical ---")
row_pct = ct_g.div(ct_g.sum(axis=1), axis=0) * 100
print(row_pct.round(1).astype(str) + "%")
for s, lab in [(0, "Not shown clinical"), (1, "Shown clinical")]:
    sub = sx[sx["shown_clinical"] == s]
    vc = sub["gender_cat"].value_counts()
    n = len(sub)
    print(f"\n{lab} (n={n:,}):")
    for cat in ["Male", "Female", "Other"]:
        k = int(vc.get(cat, 0))
        print(f"  {cat}: {k} ({100 * k / n:.1f}%)")
if ct_g.shape[0] >= 2 and ct_g.shape[1] >= 2:
    print("\nChi-square (shown_clinical × gender_cat): p =", stats.chi2_contingency(ct_g)[1])

e = df.dropna(subset=["edu_total_years", "shown_clinical"]).copy()
e0 = e.loc[e["shown_clinical"] == 0, "edu_total_years"]
e1 = e.loc[e["shown_clinical"] == 1, "edu_total_years"]
print("\n--- Education (edu_total_years) ---")
print(f"Not shown: mean {e0.mean():.2f}, SD {e0.std():.2f}, n={len(e0)}")
print(f"Shown:     mean {e1.mean():.2f}, SD {e1.std():.2f}, n={len(e1)}")
print("Welch t-test p:", stats.ttest_ind(e1, e0, equal_var=False).pvalue)

em = df.dropna(subset=["employed", "shown_clinical"]).copy()
em = em[em["employed"].isin([0.0, 1.0])]
ct_e = pd.crosstab(em["shown_clinical"], em["employed"])
print("\n--- Employment (0 = Nein, 1 = Ja) — counts ---")
print(ct_e.rename(columns={0.0: "Not employed", 1.0: "Employed"}))
print("\n--- Employment — % employed (Ja) among valid answers ---")
for s, lab in [(0, "Not shown clinical"), (1, "Shown clinical")]:
    sub = em[em["shown_clinical"] == s]
    n = len(sub)
    n_emp = int((sub["employed"] == 1.0).sum())
    print(f"{lab}: employed (Ja) = {n_emp} / {n} = {100 * n_emp / n:.1f}%")
if ct_e.shape == (2, 2):
    print("Chi-square (shown_clinical × employed): p =", stats.chi2_contingency(ct_e)[1])

=== Unique values: KI use question (first 30 by frequency) ===
ki_nutzen_sie_ki_programme_wie_z_b_chatgpt_gemini_claud
Ja      15132
Nein    13913
NaN       526
Name: count, dtype: int64

=== Analysis sample restriction (KI yes/no + gender + age) ===
  N total:                    29,571
  Excl. not Ja/Nein on KI:    526
  Excl. missing gender:       100
  Excl. missing age:          127
  (Reasons overlap; kept:)    28,910
  Final analysis N:           28,910

KI USE QUESTION × shown_clinical (raw counts) — analysis sample only
ki_answer_class     no    yes    All
shown_clinical                      
0                 6822   9086  15908
1                 7030   5972  13002
All              13852  15058  28910

--- % answering YES among those with class yes/no (excluding missing) ---
Not shown clinical: Yes = 9086 / 15908 = 57.1%
Shown clinical: Yes = 5972 / 13002 = 45.9%

--- % YES among ALL rows in group ---
Not shown clinical: Yes = 9086 / 15908 = 57.1%
Shown clinical: Yes = 5972 / 1

## Analysis sample definition (exclusions)

We restrict the analysis sample to participants with the minimum required information:

- **Age**
- **Gender**
- **AI-use response** (Yes/No)


In [10]:
# From here on, df = only cases with gender, age, and Yes/No on KI question
yes_no = ['Ja', 'ja', 'Yes', 'yes', 'Nein', 'nein', 'No', 'no']

n_total = len(df)
miss_gender = df['gender_filled'].isna()
miss_age    = df['birth_year_filled'].isna()
has_demo    = (~miss_gender) & (~miss_age)
other_ki    = has_demo & ~df[ki_col].astype(str).str.strip().isin(yes_no)

n_miss_gender = miss_gender.sum()
n_miss_age    = miss_age.sum()
n_other_ki    = other_ki.sum()

mask_keep = has_demo & df[ki_col].astype(str).str.strip().isin(yes_no)
n_excluded = n_total - mask_keep.sum()

print('Exclusions (reasons can overlap):')
print(f'  Missing gender:           {n_miss_gender:>6}')
print(f'  Missing age:              {n_miss_age:>6}')
print(f'  Non–Yes/No on KI question: {n_other_ki:>6}')
print(f'  Total excluded:           {n_excluded:>6}')
print(f'  Analysis sample (kept):   {mask_keep.sum():>6}')

df = df.loc[mask_keep].reset_index(drop=True)

Exclusions (reasons can overlap):
  Missing gender:                0
  Missing age:                   0
  Non–Yes/No on KI question:      0
  Total excluded:                0
  Analysis sample (kept):    28910


## Descriptive statistics: sociodemographics


### Age and gender


In [11]:
# Age distribution and demographics table (paper-style)

# Ensure age is computed (age as of 2026)
REFERENCE_YEAR = 2026
if 'age_2026' not in df.columns:
    df['age_2026'] = REFERENCE_YEAR - df['birth_year_filled']

age = df['age_2026']

# --- Age: full distribution ---
print("Age distribution (years, reference 2026):")
print(f"  N:        {age.count():>6}")
print(f"  Mean (SD): {age.mean():.2f} ({age.std():.2f})")
print(f"  Median:    {age.median():.2f}")
print(f"  Min–Max:   {age.min():.0f} – {age.max():.0f}")
q1, q3 = age.quantile(0.25), age.quantile(0.75)
print(f"  IQR:       {q3 - q1:.2f}  (Q1 = {q1:.2f}, Q3 = {q3:.2f})")
print()

# --- Age by bins (optional for Methods/Results) ---
bins = [18, 30, 45, 60, 75, 150]
labels = ['18–30', '31–45', '46–60', '61–75', '76+']
df['age_group'] = pd.cut(age.clip(lower=18), bins=bins, labels=labels, include_lowest=True)
age_bins = df['age_group'].value_counts().sort_index()
age_bins_pct = (df['age_group'].value_counts(normalize=True).sort_index() * 100)
print("Age groups (n, %):")
for grp in labels:
    n = age_bins.get(grp, 0)
    pct = age_bins_pct.get(grp, 0)
    print(f"  {grp:>6}: {n:>6} ({pct:.1f}%)")
print()

# --- Gender: n and % ---
print("Gender:")
gender_n = df['gender_filled'].value_counts(dropna=False)
gender_pct = df['gender_filled'].value_counts(normalize=True, dropna=False) * 100
for g in gender_n.index:
    print(f"  {g}: n = {gender_n[g]}, {gender_pct[g]:.1f}%")
print(f"  Total N: {len(df)}")
print()

# --- Compact demographics table (copy into paper) ---
print("--- Demographics table (paper format) ---")
col_name = f"Total (N = {len(df)})"
var_rows = ['Age (years)', '  M (SD)', '  Median [IQR]', '  Range', 'Gender, n (%)']
val_rows = ['', f"{age.mean():.1f} ({age.std():.1f})", f"{age.median():.0f} [{q1:.0f}–{q3:.0f}]", f"{age.min():.0f}–{age.max():.0f}", '']
for g in gender_n.index:
    var_rows.append('  ' + str(g))
    val_rows.append(f"{gender_n[g]} ({gender_pct[g]:.1f}%)")
demo = pd.DataFrame({'Variable': var_rows, col_name: val_rows})
print(demo.to_string(index=False))

Age distribution (years, reference 2026):
  N:         28910
  Mean (SD): 58.53 (14.05)
  Median:    61.00
  Min–Max:   20 – 102
  IQR:       20.00  (Q1 = 49.00, Q3 = 69.00)

Age groups (n, %):
   18–30:   1125 (3.9%)
   31–45:   4643 (16.1%)
   46–60:   8592 (29.7%)
   61–75:  11925 (41.2%)
     76+:   2625 (9.1%)

Gender:
  Weiblich: n = 17103, 59.2%
  Männlich: n = 11733, 40.6%
  Andere: n = 68, 0.2%
  Keine Angabe: n = 6, 0.0%
  Total N: 28910

--- Demographics table (paper format) ---
      Variable Total (N = 28910)
   Age (years)                  
        M (SD)       58.5 (14.1)
  Median [IQR]        61 [49–69]
         Range            20–102
 Gender, n (%)                  
      Weiblich     17103 (59.2%)
      Männlich     11733 (40.6%)
        Andere         68 (0.2%)
  Keine Angabe          6 (0.0%)


### Employment and occupation

Current employment status, sector/industry, and occupational area.


In [12]:
# Analysis sample (df): job variables

job_employed_col = 'job_sind_sie_derzeit_berufstaetig'
job_branch_col   = 'job_in_welcher_branche_arbeiten_sie_aktuell'
job_area_col     = 'job_in_welchem_berufsbereich_sind_sie_taetig'

yes_labels = ['Ja', 'ja', 'Yes', 'yes']
no_labels  = ['Nein', 'nein', 'No', 'no']

# --- 1. Currently employed (categorical distribution; % among non-missing) ---
print("job_sind_sie_derzeit_berufstaetig — distribution (analysis sample):")
employed = df[job_employed_col]

employed_counts = employed.value_counts(dropna=False)
# Percentages only among non-missing values
employed_nonmiss = employed.dropna()
employed_pct_nonmiss = employed_nonmiss.value_counts(normalize=True) * 100

for val, count in employed_counts.items():
    if pd.isna(val):
        label = '(blank)'
        print(f"  {label:<50} {count:>6}  (missing)")
    else:
        label = str(val)
        pct = employed_pct_nonmiss.get(val, 0.0)
        print(f"  {label:<50} {count:>6}  ({pct:.1f}%)")

print(f"  Total (incl. blank): {len(df)}")
print()

# --- 2. Branch (Branche) ---
print("job_in_welcher_branche_arbeiten_sie_aktuell — distribution:")
branch = df[job_branch_col]
branch_counts = branch.value_counts(dropna=False)
branch_pct_nonmiss = branch.dropna().value_counts(normalize=True) * 100
for val, count in branch_counts.items():
    if pd.isna(val):
        print(f"  {'(blank)':<50} {count:>6}  (missing)")
    else:
        pct = branch_pct_nonmiss.get(val, 0.0)
        print(f"  {str(val):<50} {count:>6}  ({pct:.1f}%)")
print(f"  Total (incl. blank): {len(df)}")
print()

# --- 3. Occupational area (Berufsbereich) ---
print("job_in_welchem_berufsbereich_sind_sie_taetig — distribution:")
area = df[job_area_col]
area_counts = area.value_counts(dropna=False)
area_pct_nonmiss = area.dropna().value_counts(normalize=True) * 100
for val, count in area_counts.items():
    if pd.isna(val):
        print(f"  {'(blank)':<50} {count:>6}  (missing)")
    else:
        pct = area_pct_nonmiss.get(val, 0.0)
        print(f"  {str(val):<50} {count:>6}  ({pct:.1f}%)")
print(f"  Total (incl. blank): {len(df)}")

job_sind_sie_derzeit_berufstaetig — distribution (analysis sample):
  (blank)                                             14199  (missing)
  Ja, ich übe eine hauptberufliche Tätigkeit aus       8441  (57.4%)
  Nein, ich bin nicht erwerbstätig                     5661  (38.5%)
  Ja, ich habe mehrere Beschäftigungsverhältnisse       450  (3.1%)
  Nein, ich bin arbeitssuchend                          159  (1.1%)
  Total (incl. blank): 28910

job_in_welcher_branche_arbeiten_sie_aktuell — distribution:
  (blank)                                             20079  (missing)
  Gesundheits- und Sozialwesen                         1903  (21.5%)
  Öffentliche Verwaltung, Verteidigung, Sozialversicherung   1548  (17.5%)
  Erziehung und Unterricht                              895  (10.1%)
  Verarbeitendes Gewerbe                                663  (7.5%)
  Erbringung von sonstigen Dienstleistungen             548  (6.2%)
  Information und Kommunikation                         530  (6.0%)
  Erbring

### Employment status and net household income


In [13]:
# Analysis sample (df): sozdem employment & income

sozdem_employed_col = 'sozdem_sind_sie_zurzeit_erwerbstaetig'
sozdem_income_col   = 'sozdem_wie_hoch_ist_das_durchschnittliche_monatliche_ne'

# --- 1. Currently employed (Ja / Nein or No / blanks / other) ---
print("sozdem_sind_sie_zurzeit_erwerbstaetig (analysis sample):")
s = df[sozdem_employed_col].astype(str).str.strip()
blanks = df[sozdem_employed_col].isna() | s.eq('') | s.str.lower().eq('nan')
starts_ja   = s.str.startswith('Ja', na=False) | s.str.startswith('ja', na=False)
starts_no   = s.str.startswith('Nein', na=False) | s.str.startswith('nein', na=False) | s.str.startswith('No', na=False) | s.str.startswith('no', na=False)

n_ja    = starts_ja.sum()
n_no    = starts_no.sum()
n_blank = blanks.sum()
n_other = len(df) - n_ja - n_no - n_blank

# Percentages only among non-missing (same as income cell below)
n_nonmiss = n_ja + n_no + n_other
pct_ja = (100.0 * n_ja / n_nonmiss) if n_nonmiss else 0.0
pct_no = (100.0 * n_no / n_nonmiss) if n_nonmiss else 0.0
pct_other = (100.0 * n_other / n_nonmiss) if n_nonmiss else 0.0

print(f"  Starts with Ja:   {n_ja:>6}  ({pct_ja:.1f}%)")
print(f"  Starts with No:   {n_no:>6}  ({pct_no:.1f}%)")
print(f"  Blank:            {n_blank:>6}  (missing)")
if n_other > 0:
    print(f"  Other:            {n_other:>6}  ({pct_other:.1f}%)")
print(f"  Total N: {len(df)}")
print()
print("Value counts (raw):")
print(df[sozdem_employed_col].value_counts(dropna=False).head(30))
print()

# --- 2. Net household income (categorical; % among non-missing) ---
print("sozdem_wie_hoch_ist_das_durchschnittliche_monatliche_ne — distribution:")
income = df[sozdem_income_col]
income_counts = income.value_counts(dropna=False)
income_pct_nonmiss = income.dropna().value_counts(normalize=True) * 100
for val, count in income_counts.items():
    if pd.isna(val):
        print(f"  {'(blank)':<50} {count:>6}  (missing)")
    else:
        pct = income_pct_nonmiss.get(val, 0.0)
        print(f"  {str(val):<50} {count:>6}  ({pct:.1f}%)")
print(f"  Total (incl. blank): {len(df)}")

sozdem_sind_sie_zurzeit_erwerbstaetig (analysis sample):
  Starts with Ja:    18459  (64.6%)
  Starts with No:    10095  (35.4%)
  Blank:               356  (missing)
  Total N: 28910

Value counts (raw):
sozdem_sind_sie_zurzeit_erwerbstaetig
Ja, Voll- oder Teilzeit erwerbstätig                                                                                        18459
Nein (einschließlich: SchülerInnen oder Studierende, die nicht gegen Geld arbeiten, VorruheständlerInnen, RentnerInnen o    10095
NaN                                                                                                                           356
Name: count, dtype: int64

sozdem_wie_hoch_ist_das_durchschnittliche_monatliche_ne — distribution:
  5 000 Euro und mehr                                  6929  (24.3%)
  3 000 bis unter 4 000 Euro                           5723  (20.0%)
  4 000 bis unter 5 000 Euro                           4867  (17.0%)
  2 250 bis unter 3 000 Euro                           4279  

### Education

Highest school degree and derived education level coding.


In [14]:
# Analysis sample (df): education variables (categorical) + years of education
edu_school_col = 'sozdem_welchen_hoechsten_allgemeinbildenden_schulabschl'
edu_coded_col  = 'education_level_coded'

# --- Mapping: Schulabschluss → Bildungsjahre (aligned to SOEP xBILZEIT) ---
school_years_map = {
    "Allgemeine oder fachgebundene Hochschulreife/Abitur (Gymnasium bzw. Abschluss der Erweiterten Oberschule (der DDR))": 13,
    "Fachhochschulreife, Abschluss einer Fachoberschule": 12,
    "Realschulabschluss (Mittlere Reife)": 10,
    "Polytechnische Oberschule der DDR mit Abschluss der 10. Klasse": 10,
    "Hauptschulabschluss (Volksschulabschluss)": 9,
    "Polytechnische Oberschule der DDR mit Abschluss der 8. oder 9. Klasse": 9,
    "von der Schule abgegangen ohne Hauptschulabschluss (Volksschulabschluss)": 7,
    "Einen anderen Schulabschluss": 10,
    "Schüler:in, besuche eine allgemeinbildende Schule (Vollzeit oder Teilzeit)": np.nan,
    "Weiß nicht": np.nan,
    "Keine Angabe": np.nan,
}

# --- Mapping: Berufsabschluss → Zusatzjahre (aligned to SOEP xBILZEIT) ---
vocational_years_map = {
    "Betriebliche Berufsausbildung (Lehre) abgeschlossen": 1.5,
    "Schulische Berufsausbildung abgeschlossen (Berufsfachschule, Handelsschule etc.)": 2.0,
    "Ausbildung an einer Fach-, Meister-, Technikerschule, Berufs- oder Fachakademie abgeschlossen": 3,
    "Ausbildung an einer Fachschule der DDR abgeschlossen": 3,
    "Bachelor an einer (Fach-) Hochschule abgeschlossen": 5,
    "Fachhochschulabschluss (z.B. Diplom, Master)": 5,
    "Universitätsabschluss (z.B. Diplom, Magister, Staatsexamen, Master)": 5,
    "Promotion": 8,
    "Keinen beruflichen Abschluss und nicht in beruflicher Ausbildung": 0,
    "Noch in beruflicher Ausbildung (Berufsvorbereitungsjahr, Ausbildung, Praktikum, Studium)": 0,
    "Schüler/in und besuche eine berufsorientierte Aufbau-, Fachschule o.ä.": 0,
    "Einen anderen beruflichen Abschluss": np.nan,
    "Weiß nicht": np.nan,
    "Keine Angabe": np.nan,
}

# --- Compute years (SOEP xBILZEIT logic) ---
df['edu_school_years']     = df[edu_school_col].map(school_years_map)
voc_str                     = df[edu_coded_col].astype(str).str.strip()
df['edu_vocational_years'] = voc_str.map(vocational_years_map)

df['edu_total_years'] = np.nan
mask_add = df['edu_school_years'].notna() & (df['edu_school_years'] >= 7)
df.loc[mask_add, 'edu_total_years'] = (
    df.loc[mask_add, 'edu_school_years'] +
    df.loc[mask_add, 'edu_vocational_years']
)

# --- 1. Highest general school degree ---
print("sozdem_welchen_hoechsten_allgemeinbildenden_schulabschl — distribution:")
school             = df[edu_school_col]
school_counts      = school.value_counts(dropna=False)
school_pct_nonmiss = school.dropna().value_counts(normalize=True) * 100
for val, count in school_counts.items():
    if pd.isna(val):
        print(f"  {'(blank)':<50} {count:>6}  (missing)")
    else:
        pct = school_pct_nonmiss.get(val, 0.0)
        print(f"  {str(val):<50} {count:>6}  ({pct:.1f}%)")
print(f"  Total (incl. blank): {len(df)}")
print()

# --- 2. Education level (coded) ---
print("education_level_coded — distribution:")
coded             = df[edu_coded_col]
coded_counts      = coded.value_counts(dropna=False)
coded_pct_nonmiss = coded.dropna().value_counts(normalize=True) * 100
for val, count in coded_counts.items():
    if pd.isna(val):
        print(f"  {'(blank)':<50} {count:>6}  (missing)")
    else:
        pct = coded_pct_nonmiss.get(val, 0.0)
        print(f"  {str(val):<50} {count:>6}  ({pct:.1f}%)")
print(f"  Total (incl. blank): {len(df)}")

# --- 3. Years of education ---
years_valid = df['edu_total_years'].dropna()
n_valid     = len(years_valid)
n_miss      = len(df) - n_valid

print()
print("Years of education (Schulabschluss + Berufsabschluss Zusatzjahre):")
print(f"  N with valid years:                                  {n_valid:>6}")
print(f"  N missing (NA in school and/or vocational mapping):  {n_miss:>6}")
if n_valid > 0:
    print(f"  Mean:    {years_valid.mean():.2f} years")
    print(f"  Std:     {years_valid.std():.2f} years")
    print(f"  Median:  {years_valid.median():.2f} years")
    print(f"  Min–Max: {years_valid.min():.0f} – {years_valid.max():.0f}")
    print()
    print("  Distribution (counts):")
    for yrs, count in years_valid.value_counts().sort_index().items():
        print(f"    {yrs:>5} years: {count:>6}")


sozdem_welchen_hoechsten_allgemeinbildenden_schulabschl — distribution:
  Allgemeine oder fachgebundene Hochschulreife/Abitur (Gymnasium bzw. Abschluss der Erweiterten Oberschule (der DDR))  15637  (54.7%)
  Realschulabschluss (Mittlere Reife)                  4388  (15.3%)
  Polytechnische Oberschule der DDR mit Abschluss der 10. Klasse   3646  (12.7%)
  Fachhochschulreife, Abschluss einer Fachoberschule   3427  (12.0%)
  Hauptschulabschluss (Volksschulabschluss)            1042  (3.6%)
  (blank)                                               312  (missing)
  Einen anderen Schulabschluss                          189  (0.7%)
  Polytechnische Oberschule der DDR mit Abschluss der 8. oder 9. Klasse     95  (0.3%)
  Keine Angabe                                           62  (0.2%)
  von der Schule abgegangen ohne Hauptschulabschluss (Volksschulabschluss)     52  (0.2%)
  Schüler:in, besuche eine allgemeinbildende Schule (Vollzeit oder Teilzeit)     49  (0.2%)
  Weiß nicht                   

### Household size


In [15]:
# Analysis sample (df): household size

hh_size_col = 'sozdem_wie_viele_personen_leben_in_ihrem_haushalt'

print("sozdem_wie_viele_personen_leben_in_ihrem_haushalt — distribution (household size, categorical):")
hh = df[hh_size_col]
hh_counts = hh.value_counts(dropna=False)
hh_pct_nonmiss = hh.dropna().value_counts(normalize=True) * 100

for val, count in hh_counts.items():
    if pd.isna(val):
        print(f"  {'(blank)':<20} {count:>6}  (missing)")
    else:
        pct = hh_pct_nonmiss.get(val, 0.0)
        print(f"  {str(val):<20} {count:>6}  ({pct:.1f}%)")

print(f"  Total (incl. blank): {len(df)}")

sozdem_wie_viele_personen_leben_in_ihrem_haushalt — distribution (household size, categorical):
  2                     15245  (53.4%)
  1                      5507  (19.3%)
  3                      3868  (13.5%)
  4                      3032  (10.6%)
  5                       690  (2.4%)
  (blank)                 344  (missing)
  mehr als 5              224  (0.8%)
  Total (incl. blank): 28910


### Social contacts (loneliness-related item)

Number of personal contacts in the last month.


In [16]:
# Übersicht zur Variable loneliness_9 (Anzahl der Kontakte im letzten Monat)

loneliness_col = 'loneliness_9'

print("loneliness_9 — Übersicht (Anzahl der Kontakte im letzten Monat):")
loneliness_vals = df[loneliness_col]
loneliness_counts = loneliness_vals.value_counts(dropna=False).sort_index()
loneliness_nonmiss = loneliness_vals.replace('', np.nan).replace('nan', np.nan).dropna().astype(float)
n_total = len(df)
n_valid = len(loneliness_nonmiss)
n_blanks = n_total - n_valid

print(f"  N mit gültigen Angaben: {n_valid:>6}")
print(f"  N missing / blank:     {n_blanks:>6}")

if n_valid > 0:
    print(f"  Mittelwert (mean):    {loneliness_nonmiss.mean():.2f}")
    print(f"  Median:               {loneliness_nonmiss.median():.2f}")
    print(f"  Min–Max:              {loneliness_nonmiss.min():.0f} – {loneliness_nonmiss.max():.0f}")
    print(f"  Standardabweichung:   {loneliness_nonmiss.std():.2f}")
    print(f"  25./75. Perzentil:    {loneliness_nonmiss.quantile(.25):.2f} / {loneliness_nonmiss.quantile(.75):.2f}")
    print("\n  Verteilung (counts):")
    for val, count in loneliness_counts.items():
        if pd.isna(val) or str(val).strip() == '' or str(val) == 'nan':
            print(f"    {'(blank)':<10} {count:>6}  (missing)")
        else:
            print(f"    {str(val):<10} {count:>6}")
else:
    print("  (Keine gültigen Angaben vorhanden)")


loneliness_9 — Übersicht (Anzahl der Kontakte im letzten Monat):
  N mit gültigen Angaben:   4271
  N missing / blank:      24639
  Mittelwert (mean):    4.12
  Median:               4.00
  Min–Max:              1 – 6
  Standardabweichung:   1.09
  25./75. Perzentil:    4.00 / 5.00

  Verteilung (counts):
    1.0            87
    2.0           253
    3.0           654
    4.0          1714
    5.0          1186
    6.0           377
    (blank)     24639  (missing)


### Self-rated health status


In [17]:
# Untersuchung der Variable MEHM_GZ (Selbsteinschätzung des Gesundheitszustands)
mehm_col = 'MEHM_GZ'

label_mehm = {
    'A': 'Sehr gut',
    'B': 'Gut',
    'C': 'Zufriedenstellend',
    'D': 'Weniger zufriedenstellend',
    'E': 'Schlecht'
}

print("MEHM_GZ — Selbsteinschätzung des Gesundheitszustands (A=Sehr gut, B=Gut, C=Zufriedenstellend, D=Weniger zufriedenstellend, E=Schlecht):")
mehm_vals = df[mehm_col].astype(str).str.strip()
mehm_counts = mehm_vals.value_counts(dropna=False)
mehm_valid = mehm_vals.replace('', np.nan).replace('nan', np.nan).dropna()
mehm_pct = mehm_valid.value_counts(normalize=True) * 100 if len(mehm_valid) > 0 else pd.Series(dtype=float)

n_total = len(df)
n_valid = len(mehm_valid)
n_blanks = n_total - n_valid

print(f"  N mit gültigen Angaben: {n_valid:>6}")
print(f"  N missing / blank:     {n_blanks:>6}\n")

print("  Verteilung der Antworten:")
for val, count in mehm_counts.items():
    if pd.isna(val) or str(val).strip() == '' or str(val).lower() == 'nan':
        print(f"    {'(blank)':<30} {count:>6}  (missing)")
    else:
        lbl = label_mehm.get(val.upper(), val)
        pct = mehm_pct.get(val, 0.0)
        print(f"    {lbl:<30} {count:>6}  ({pct:.1f}%)")

print(f"  Total (incl. blank): {len(df)}")


MEHM_GZ — Selbsteinschätzung des Gesundheitszustands (A=Sehr gut, B=Gut, C=Zufriedenstellend, D=Weniger zufriedenstellend, E=Schlecht):
  N mit gültigen Angaben:  12970
  N missing / blank:      15940

  Verteilung der Antworten:
    (blank)                         15940  (missing)
    Gut                              6138  (47.3%)
    Zufriedenstellend                3770  (29.1%)
    Sehr gut                         1499  (11.6%)
    Weniger zufriedenstellend        1358  (10.5%)
    Schlecht                          205  (1.6%)
  Total (incl. blank): 28910


In [18]:
eq5d_col = 'EQ5D5L_3'

label_map = {
    'a': 'Ich habe keine Probleme, meinen alltäglichen Tätigkeiten nachzugehen.',
    'b': 'Ich habe leichte Probleme, meinen alltäglichen Tätigkeiten nachzugehen.',
    'c': 'Ich habe mäßige Probleme, meinen alltäglichen Tätigkeiten nachzugehen.',
    'd': 'Ich habe große Probleme, meinen alltäglichen Tätigkeiten nachzugehen.',
    'e': 'Ich bin nicht in der Lage, meinen alltäglichen Tätigkeiten nachzugehen.'
}

valid_letters = ['a', 'b', 'c', 'd', 'e']

s = df[eq5d_col]

# robust missing detection (NaN/blank/'nan' strings)
v_str = s.astype(str).str.strip()
v_norm = v_str.str.lower()
missing_mask = s.isna() | v_str.eq('') | v_norm.isin(['nan', 'none', '<na>'])

n_total = len(df)
n_missing = int(missing_mask.sum())
n_answered = n_total - n_missing

print('EQ5D5L_3 — distribution (a-e): problems with daily activities')
print(f'  Total N:   {n_total}')
print(f'  Missing:   {n_missing}')
print(f'  Answered:  {n_answered}')

if n_answered == 0:
    print('  No answered values (cannot compute category distribution).')
else:
    print('  Counts among answered (and %):')
    for letter in valid_letters:
        n_cat = int((~missing_mask & v_norm.eq(letter)).sum())
        pct_answered = 100.0 * n_cat / n_answered
        pct_total = 100.0 * n_cat / n_total
        print(
            f"    {letter} = {label_map[letter]} : {n_cat:>6} "
            f"({pct_answered:5.1f}% of answered; {pct_total:5.1f}% of total)"
        )

    # optional sanity check: unexpected non-missing values
    n_unexpected = int((~missing_mask & ~v_norm.isin(valid_letters)).sum())
    if n_unexpected > 0:
        print(f'  Unexpected (not in a-e): {n_unexpected}')
        print(
            '  Unexpected values (top 10):',
            v_norm[~missing_mask & ~v_norm.isin(valid_letters)].value_counts().head(10).to_dict()
        )

EQ5D5L_3 — distribution (a-e): problems with daily activities
  Total N:   28910
  Missing:   15935
  Answered:  12975
  Counts among answered (and %):
    a = Ich habe keine Probleme, meinen alltäglichen Tätigkeiten nachzugehen. :   6583 ( 50.7% of answered;  22.8% of total)
    b = Ich habe leichte Probleme, meinen alltäglichen Tätigkeiten nachzugehen. :   3910 ( 30.1% of answered;  13.5% of total)
    c = Ich habe mäßige Probleme, meinen alltäglichen Tätigkeiten nachzugehen. :   1815 ( 14.0% of answered;   6.3% of total)
    d = Ich habe große Probleme, meinen alltäglichen Tätigkeiten nachzugehen. :    592 (  4.6% of answered;   2.0% of total)
    e = Ich bin nicht in der Lage, meinen alltäglichen Tätigkeiten nachzugehen. :     75 (  0.6% of answered;   0.3% of total)


## Descriptive statistics: clinical variables


### Treatment and diagnosis

Current treatment, self-reported diagnosis, and diagnosis prevalence.


In [19]:
# Analysis sample (df): treatment, self-reported diagnosis, diagnosis prevalence

behandlung_col = 'Behandlung_dich'
diagnose_selbst_col = 'Diagnose_selbst_dich'

# Label mappings: a=Ja, b=No, c=Weiß nicht (only for Diagnose_selbst_dich)
label_behandlung = {'a': 'Ja', 'b': 'Nein'}
label_diagnose_selbst = {'a': 'Ja', 'b': 'Nein', 'c': "Weiß nicht / I don't know"}

# --- 1. Behandlung_dich (a=Ja, b=No) ---
print("Behandlung_dich — distribution (a = Ja, b = Nein):")
bh = df[behandlung_col].astype(str).str.strip()
bh_counts = bh.value_counts(dropna=False)
bh_valid = bh.replace('', np.nan).replace('nan', np.nan).dropna()
bh_pct = bh_valid.value_counts(normalize=True) * 100 if len(bh_valid) > 0 else pd.Series(dtype=float)
for val, count in bh_counts.items():
    if pd.isna(val) or str(val).strip() == '' or str(val) == 'nan':
        print(f"  {'(blank)':<30} {count:>6}  (missing)")
    else:
        lbl = label_behandlung.get(val, val)
        pct = bh_pct.get(val, 0.0)
        print(f"  {lbl:<30} {count:>6}  ({pct:.1f}%)")
print(f"  Total (incl. blank): {len(df)}")
print()

# --- 2. Diagnose_selbst_dich (a=Ja, b=No, c=Weiß nicht) ---
print("Diagnose_selbst_dich — distribution (a = Ja, b = Nein, c = Weiß nicht):")
ds = df[diagnose_selbst_col].astype(str).str.strip()
ds_counts = ds.value_counts(dropna=False)
ds_valid = ds.replace('', np.nan).replace('nan', np.nan).dropna()
ds_pct = ds_valid.value_counts(normalize=True) * 100
for val, count in ds_counts.items():
    if pd.isna(val) or str(val).strip() == '' or str(val) == 'nan':
        print(f"  {'(blank)':<30} {count:>6}  (missing)")
    else:
        lbl = label_diagnose_selbst.get(val, val)
        pct = ds_pct.get(val, 0.0)
        print(f"  {lbl:<30} {count:>6}  ({pct:.1f}%)")
print(f"  Total (incl. blank): {len(df)}")
print()

# --- 3. Prevalence of specific diagnoses (among those who answered 'a' = Ja to Diagnose_selbst_dich) ---
# Only people who answered 'a' (Ja) get the follow-up; they answer 0 = not present, 1 = present per diagnosis.
diagnose_cols = [
    'Diagnose_PD', 'Diagnose_DD', 'Diagnose_BD', 'Diagnose_AD', 'Diagnose_OCD',
    'Diagnose_PTSD', 'Diagnose_SD', 'Diagnose_ED', 'Diagnose_PBD', 'Diagnose_DMD',
    'Diagnose_ADHD', 'Diagnose_ASD', 'Diagnose_SUD', 'Diagnose_D', 'Diagnose_andere'
]

mask_has_diagnose = (df[diagnose_selbst_col].astype(str).str.strip().str.lower() == 'a')
df_with_diagnose = df.loc[mask_has_diagnose]
n_with_diagnose = len(df_with_diagnose)

print("Diagnosis prevalence (among those who answered 'a' = Ja to Diagnose_selbst_dich):")
print(f"  N with Diagnose_selbst_dich = Ja: {n_with_diagnose}")
print()
if n_with_diagnose > 0:
    print(f"  {'Diagnosis':<25} {'n':>8}  {'% of subgroup':>14}")
    print("  " + "-" * 50)
    for col in diagnose_cols:
        if col not in df.columns:
            print(f"  {col:<25} (column missing)")
            continue
        # 0 = not present, 1 = present (numeric or string)
        vals = df_with_diagnose[col]
        present = (pd.to_numeric(vals, errors='coerce') == 1) | (vals.astype(str).str.strip() == '1')
        n_yes = present.sum()
        pct = 100 * n_yes / n_with_diagnose
        print(f"  {col:<25} {n_yes:>8}  {pct:>13.1f}%")
else:
    print("  No one answered 'a' (Ja) to Diagnose_selbst_dich; prevalence not computed.")

Behandlung_dich — distribution (a = Ja, b = Nein):
  (blank)                         15993  (missing)
  Nein                            11869  (91.9%)
  Ja                               1048  (8.1%)
  Total (incl. blank): 28910

Diagnose_selbst_dich — distribution (a = Ja, b = Nein, c = Weiß nicht):
  (blank)                         15965  (missing)
  Nein                            10203  (78.8%)
  Ja                               2097  (16.2%)
  Weiß nicht / I don't know         645  (5.0%)
  Total (incl. blank): 28910

Diagnosis prevalence (among those who answered 'a' = Ja to Diagnose_selbst_dich):
  N with Diagnose_selbst_dich = Ja: 2097

  Diagnosis                        n   % of subgroup
  --------------------------------------------------
  Diagnose_PD                     67            3.2%
  Diagnose_DD                   1549           73.9%
  Diagnose_BD                     46            2.2%
  Diagnose_AD                    710           33.9%
  Diagnose_OCD                

In [20]:
import numpy as np
import pandas as pd

# Analysis sample (df): treatment, self-reported diagnosis, diagnosis prevalence

behandlung_col = 'Behandlung_dich'
diagnose_selbst_col = 'Diagnose_selbst_dich'

# Label mappings: a=Ja, b=No, c=Weiß nicht (only for Diagnose_selbst_dich)
label_behandlung = {'a': 'Ja', 'b': 'Nein'}
label_diagnose_selbst = {'a': 'Ja', 'b': 'Nein', 'c': "Weiß nicht / I don't know"}

# --- 1. Behandlung_dich (a=Ja, b=No) ---
print("Behandlung_dich — distribution (a = Ja, b = Nein):")
bh = df[behandlung_col].astype(str).str.strip()
bh_counts = bh.value_counts(dropna=False)
bh_valid = bh.replace('', np.nan).replace('nan', np.nan).dropna()
bh_pct = bh_valid.value_counts(normalize=True) * 100 if len(bh_valid) > 0 else pd.Series(dtype=float)
for val, count in bh_counts.items():
    if pd.isna(val) or str(val).strip() == '' or str(val) == 'nan':
        print(f"  {'(blank)':<30} {count:>6}  (missing)")
    else:
        lbl = label_behandlung.get(val, val)
        pct = bh_pct.get(val, 0.0)
        print(f"  {lbl:<30} {count:>6}  ({pct:.1f}%)")
print(f"  Total (incl. blank): {len(df)}")
print()

# --- 2. Diagnose_selbst_dich (a=Ja, b=No, c=Weiß nicht) ---
print("Diagnose_selbst_dich — distribution (a = Ja, b = Nein, c = Weiß nicht):")
ds = df[diagnose_selbst_col].astype(str).str.strip()
ds_counts = ds.value_counts(dropna=False)
ds_valid = ds.replace('', np.nan).replace('nan', np.nan).dropna()
ds_pct = ds_valid.value_counts(normalize=True) * 100
for val, count in ds_counts.items():
    if pd.isna(val) or str(val).strip() == '' or str(val) == 'nan':
        print(f"  {'(blank)':<30} {count:>6}  (missing)")
    else:
        lbl = label_diagnose_selbst.get(val, val)
        pct = ds_pct.get(val, 0.0)
        print(f"  {lbl:<30} {count:>6}  ({pct:.1f}%)")
print(f"  Total (incl. blank): {len(df)}")
print()

# --- 3. Prevalence of specific diagnoses (among those who answered 'a' = Ja to Diagnose_selbst_dich) ---
# Only people who answered 'a' (Ja) get the follow-up; they answer 0 = not present, 1 = present per diagnosis.
diagnose_cols = [
    'Diagnose_PD', 'Diagnose_DD', 'Diagnose_BD', 'Diagnose_AD', 'Diagnose_OCD',
    'Diagnose_PTSD', 'Diagnose_SD', 'Diagnose_ED', 'Diagnose_PBD', 'Diagnose_DMD',
    'Diagnose_ADHD', 'Diagnose_ASD', 'Diagnose_SUD', 'Diagnose_D', 'Diagnose_andere'
]

mask_has_diagnose = (df[diagnose_selbst_col].astype(str).str.strip().str.lower() == 'a')
df_with_diagnose = df.loc[mask_has_diagnose]
n_with_diagnose = len(df_with_diagnose)

# --- 3b. n_diagnoses: Ja → sum Diagnose_* == 1; Nein/Weiß nicht/anderes (nicht leer) → 0; leer → NaN ---
ds_self = df[diagnose_selbst_col]
ds_stripped = ds_self.astype(str).str.strip()
ds_lower = ds_stripped.str.lower()
is_blank = (
    ds_self.isna()
    | (ds_stripped == '')
    | ds_lower.isin(['nan', 'none'])
)
mask_yes = ds_lower == 'a'
mask_other_answered = (~is_blank) & (~mask_yes)

df['n_diagnoses'] = np.nan
df.loc[mask_other_answered, 'n_diagnoses'] = 0

cols_ok = [c for c in diagnose_cols if c in df.columns]
missing_cols = [c for c in diagnose_cols if c not in df.columns]
if missing_cols:
    print(f"Hinweis: fehlende Diagnose-Spalten (für n_diagnoses ignoriert): {missing_cols}")
    print()

if cols_ok and mask_yes.any():
    blk = df.loc[mask_yes, cols_ok]
    present = pd.DataFrame(
        {
            c: (pd.to_numeric(blk[c], errors='coerce') == 1)
            | (blk[c].astype(str).str.strip() == '1')
            for c in cols_ok
        },
        index=blk.index,
    )
    df.loc[mask_yes, 'n_diagnoses'] = present.sum(axis=1)

print("Diagnosis prevalence (among those who answered 'a' = Ja to Diagnose_selbst_dich):")
print(f"  N with Diagnose_selbst_dich = Ja: {n_with_diagnose}")
print()
if n_with_diagnose > 0:
    print(f"  {'Diagnosis':<25} {'n':>8}  {'% of subgroup':>14}")
    print("  " + "-" * 50)
    for col in diagnose_cols:
        if col not in df.columns:
            print(f"  {col:<25} (column missing)")
            continue
        vals = df_with_diagnose[col]
        present = (pd.to_numeric(vals, errors='coerce') == 1) | (vals.astype(str).str.strip() == '1')
        n_yes = present.sum()
        pct = 100 * n_yes / n_with_diagnose
        print(f"  {col:<25} {n_yes:>8}  {pct:>13.1f}%")
    print()
    print("  n_diagnoses (Ja = sum of Diagnose_* == 1; answered but not Ja = 0; blank = missing):")
    vc = df['n_diagnoses'].value_counts(dropna=False).sort_index()
    for k, v in vc.items():
        if pd.isna(k):
            print(f"    {'(missing)':<12} {int(v):>6}")
        else:
            print(f"    {int(k):<12} {int(v):>6}")
    print(f"    mean (non-missing): {df['n_diagnoses'].mean():.2f}")
else:
    print("  No one answered 'a' (Ja) to Diagnose_selbst_dich; prevalence not computed.")
    print()
    print("  n_diagnoses (Ja = sum of Diagnose_* == 1; answered but not Ja = 0; blank = missing):")
    vc = df['n_diagnoses'].value_counts(dropna=False).sort_index()
    for k, v in vc.items():
        if pd.isna(k):
            print(f"    {'(missing)':<12} {int(v):>6}")
        else:
            print(f"    {int(k):<12} {int(v):>6}")
    print(f"    mean (non-missing): {df['n_diagnoses'].mean():.2f}")

Behandlung_dich — distribution (a = Ja, b = Nein):
  (blank)                         15993  (missing)
  Nein                            11869  (91.9%)
  Ja                               1048  (8.1%)
  Total (incl. blank): 28910

Diagnose_selbst_dich — distribution (a = Ja, b = Nein, c = Weiß nicht):
  (blank)                         15965  (missing)
  Nein                            10203  (78.8%)
  Ja                               2097  (16.2%)
  Weiß nicht / I don't know         645  (5.0%)
  Total (incl. blank): 28910

Diagnosis prevalence (among those who answered 'a' = Ja to Diagnose_selbst_dich):
  N with Diagnose_selbst_dich = Ja: 2097

  Diagnosis                        n   % of subgroup
  --------------------------------------------------
  Diagnose_PD                     67            3.2%
  Diagnose_DD                   1549           73.9%
  Diagnose_BD                     46            2.2%
  Diagnose_AD                    710           33.9%
  Diagnose_OCD                

### Psychological treatment: lifetime vs. current


In [21]:
# Counts for Behandlung_dich (current psychological treatment) and Erstkontakt_nein (ever in psychological treatment)

behandlung_col = 'Behandlung_dich'
erstkontakt_col = 'Erstkontakt_nein'

# Label mappings: a=Ja, b=Nein
label_map = {'a': 'Ja', 'b': 'Nein'}

# --- Behandlung_dich (current treatment) ---
print("Aktuelle Behandlung (a = Ja, b = Nein):")
bh = df[behandlung_col].astype(str).str.strip()
bh_blank = bh.isna() | bh.eq('') | bh.str.lower().eq('nan')
bh_valid = bh[~bh_blank]
bh_counts = bh.value_counts(dropna=False)
bh_pct = bh_valid.value_counts(normalize=True) * 100 if len(bh_valid) > 0 else pd.Series(dtype=float)
for val, count in bh_counts.items():
    if pd.isna(val) or str(val).strip() == '' or str(val).strip().lower() == 'nan':
        print(f"  {'(blank)':<30} {count:>6}  (missing)")
    else:
        lbl = label_map.get(val, val)
        pct = bh_pct.get(val, 0.0)
        print(f"  {lbl:<30} {count:>6}  ({pct:.1f}%)")
print(f"  Total (incl. blank): {len(df)}")
print(f"  N valid (for %): {len(bh_valid)}")
print()

# --- Erstkontakt_nein (ever in psychological treatment) ---
print("Jemals Behandlung (a = Ja, b = Nein):")
ek = df[erstkontakt_col].astype(str).str.strip()
ek_blank = ek.isna() | ek.eq('') | ek.str.lower().eq('nan')
ek_valid = ek[~ek_blank]
ek_counts = ek.value_counts(dropna=False)
ek_pct = ek_valid.value_counts(normalize=True) * 100 if len(ek_valid) > 0 else pd.Series(dtype=float)
for val, count in ek_counts.items():
    if pd.isna(val) or str(val).strip() == '' or str(val).strip().lower() == 'nan':
        print(f"  {'(blank)':<30} {count:>6}  (missing)")
    else:
        lbl = label_map.get(val, val)
        pct = ek_pct.get(val, 0.0)
        print(f"  {lbl:<30} {count:>6}  ({pct:.1f}%)")
print(f"  Total (incl. blank): {len(df)}")
print(f"  N valid (for %): {len(ek_valid)}")


Aktuelle Behandlung (a = Ja, b = Nein):
  (blank)                         15993  (missing)
  Nein                            11869  (91.9%)
  Ja                               1048  (8.1%)
  Total (incl. blank): 28910
  N valid (for %): 12917

Jemals Behandlung (a = Ja, b = Nein):
  (blank)                         15908  (missing)
  1.0                              8295  (63.8%)
  0.0                              4707  (36.2%)
  Total (incl. blank): 28910
  N valid (for %): 13002


### Suicide attempt (self-report)


In [22]:
# Analysis: Suizidversuch (a = Yes, b = No, c = I don't know)

suizidversuch_col = 'Suizidversuch'
label_suizidversuch = {'a': 'Yes', 'b': 'No', 'c': "I don't know"}

print("Suizidversuch — distribution (a = Yes, b = No, c = I don't know):")
sv = df[suizidversuch_col].astype(str).str.strip()
sv_counts = sv.value_counts(dropna=False)
sv_valid = sv.replace('', np.nan).replace('nan', np.nan).dropna()
sv_pct = sv_valid.value_counts(normalize=True) * 100 if len(sv_valid) > 0 else pd.Series(dtype=float)
for val, count in sv_counts.items():
    if pd.isna(val) or str(val).strip() == '' or str(val) == 'nan':
        print(f"  {'(blank)':<30} {count:>6}  (missing)")
    else:
        lbl = label_suizidversuch.get(val.lower(), val)
        pct = sv_pct.get(val, 0.0)
        print(f"  {lbl:<30} {count:>6}  ({pct:.1f}%)")
print(f"  Total (incl. blank): {len(df)}")
print()
print("Value counts (raw):")
print(df[suizidversuch_col].value_counts(dropna=False))

Suizidversuch — distribution (a = Yes, b = No, c = I don't know):
  (blank)                         15921  (missing)
  No                              12200  (93.9%)
  Yes                               534  (4.1%)
  I don't know                      255  (2.0%)
  Total (incl. blank): 28910

Value counts (raw):
Suizidversuch
NaN    15921
b      12200
a        534
c        255
Name: count, dtype: int64


## Psychometric scale scores

This section computes scale scores and summarizes their distributions in the analysis sample.


### HiTOP: total sum score


In [23]:
# HiTOP sum score: recode a,b,c,d,e → 1,2,3,4,5 then sum

hitop_cols = [f'HiTOP_{i}' for i in range(1, 29)]
hitop_cols = [c for c in hitop_cols if c in df.columns]
# Map letter responses to numeric (a=1, b=2, c=3, d=4, e=5)
letter_to_num = {'a': 1, 'b': 2, 'c': 3, 'd': 4, 'e': 5}

def recode_letter(s):
    s = str(s).strip().lower()
    return letter_to_num.get(s, np.nan)

# Recode each HiTOP item and sum
hitop_numeric = df[hitop_cols].apply(lambda col: col.astype(str).str.strip().str.lower().map(letter_to_num))
df['HiTOP_sum'] = hitop_numeric.sum(axis=1)
mask_complete = hitop_numeric.notna().all(axis=1)
df.loc[~mask_complete, 'HiTOP_sum'] = np.nan

hitop_sum = df['HiTOP_sum'].dropna()
n_valid = len(hitop_sum)
n_miss = len(df) - n_valid

print("HiTOP sum score (HiTOP_1 + … + HiTOP_28), recoded a=1, b=2, c=3, d=4, e=5:")
print(f"  N with complete data: {n_valid:>6}")
print(f"  N missing (any item): {n_miss:>6}")
print()
if n_valid > 0:
    print("  Distribution:")
    print(f"    Mean (SD):   {hitop_sum.mean():.2f} ({hitop_sum.std():.2f})")
    print(f"    Median:     {hitop_sum.median():.2f}")
    print(f"    Min – Max:  {hitop_sum.min():.0f} – {hitop_sum.max():.0f}")
    q1, q3 = hitop_sum.quantile(0.25), hitop_sum.quantile(0.75)
    print(f"    IQR:        {q3 - q1:.2f}  (Q1 = {q1:.2f}, Q3 = {q3:.2f})")
    print()
    print("  Value counts:")
    for val, count in hitop_sum.value_counts().sort_index().items():
        pct = 100 * count / n_valid
        print(f"    {val:>3.0f}: {count:>6}  ({pct:.1f}%)")

HiTOP sum score (HiTOP_1 + … + HiTOP_28), recoded a=1, b=2, c=3, d=4, e=5:
  N with complete data:  12604
  N missing (any item):  16306

  Distribution:
    Mean (SD):   49.56 (10.36)
    Median:     48.00
    Min – Max:  28 – 115
    IQR:        13.00  (Q1 = 42.00, Q3 = 55.00)

  Value counts:
     28:      8  (0.1%)
     29:      4  (0.0%)
     30:     20  (0.2%)
     31:     17  (0.1%)
     32:     58  (0.5%)
     33:     96  (0.8%)
     34:    147  (1.2%)
     35:    189  (1.5%)
     36:    230  (1.8%)
     37:    299  (2.4%)
     38:    390  (3.1%)
     39:    412  (3.3%)
     40:    453  (3.6%)
     41:    493  (3.9%)
     42:    551  (4.4%)
     43:    514  (4.1%)
     44:    584  (4.6%)
     45:    512  (4.1%)
     46:    550  (4.4%)
     47:    552  (4.4%)
     48:    533  (4.2%)
     49:    474  (3.8%)
     50:    521  (4.1%)
     51:    447  (3.5%)
     52:    428  (3.4%)
     53:    420  (3.3%)
     54:    357  (2.8%)
     55:    312  (2.5%)
     56:    313  (2.5%)
     57

### HiTOP: general severity (items 1–8)


In [24]:
# HiTOP_1 - HiTOP_8 = Genereller Schweregrad sum score (reduced 1–8 only)
hitop_gs_cols = [f'HiTOP_{i}' for i in range(1, 9) if f'HiTOP_{i}' in df.columns]
# Map letter responses to numeric (a=1, b=2, c=3, d=4, e=5), same as before
hitop_gs_numeric = df[hitop_gs_cols].apply(lambda col: col.astype(str).str.strip().str.lower().map(letter_to_num))
df['HiTOP_GS_sum'] = hitop_gs_numeric.sum(axis=1)
gs_mask_complete = hitop_gs_numeric.notna().all(axis=1)
df.loc[~gs_mask_complete, 'HiTOP_GS_sum'] = np.nan

hitop_gs_sum = df['HiTOP_GS_sum'].dropna()
n_valid_gs = len(hitop_gs_sum)
n_miss_gs = len(df) - n_valid_gs

print("HiTOP Genereller Schweregrad (HiTOP_1 – HiTOP_8), recoded a=1, b=2, c=3, d=4, e=5:")
print(f"  N with complete data: {n_valid_gs:>6}")
print(f"  N missing (any item): {n_miss_gs:>6}")
print()
if n_valid_gs > 0:
    print("  Distribution:")
    print(f"    Mean (SD):   {hitop_gs_sum.mean():.2f} ({hitop_gs_sum.std():.2f})")
    print(f"    Median:     {hitop_gs_sum.median():.2f}")
    print(f"    Min – Max:  {hitop_gs_sum.min():.0f} – {hitop_gs_sum.max():.0f}")
    q1_gs, q3_gs = hitop_gs_sum.quantile(0.25), hitop_gs_sum.quantile(0.75)
    print(f"    IQR:        {q3_gs - q1_gs:.2f}  (Q1 = {q1_gs:.2f}, Q3 = {q3_gs:.2f})")
    print()
    print("  Value counts:")
    for val, count in hitop_gs_sum.value_counts().sort_index().items():
        pct = 100 * count / n_valid_gs
        print(f"    {val:>3.0f}: {count:>6}  ({pct:.1f}%)")

HiTOP Genereller Schweregrad (HiTOP_1 – HiTOP_8), recoded a=1, b=2, c=3, d=4, e=5:
  N with complete data:  12880
  N missing (any item):  16030

  Distribution:
    Mean (SD):   15.64 (4.99)
    Median:     15.00
    Min – Max:  8 – 40
    IQR:        6.00  (Q1 = 12.00, Q3 = 18.00)

  Value counts:
      8:    206  (1.6%)
      9:    433  (3.4%)
     10:    893  (6.9%)
     11:   1146  (8.9%)
     12:   1240  (9.6%)
     13:   1189  (9.2%)
     14:   1141  (8.9%)
     15:   1071  (8.3%)
     16:   1028  (8.0%)
     17:    810  (6.3%)
     18:    696  (5.4%)
     19:    583  (4.5%)
     20:    458  (3.6%)
     21:    369  (2.9%)
     22:    311  (2.4%)
     23:    259  (2.0%)
     24:    214  (1.7%)
     25:    186  (1.4%)
     26:    146  (1.1%)
     27:    100  (0.8%)
     28:    112  (0.9%)
     29:     76  (0.6%)
     30:     53  (0.4%)
     31:     47  (0.4%)
     32:     34  (0.3%)
     33:     33  (0.3%)
     34:     14  (0.1%)
     35:     13  (0.1%)
     36:      9  (0.1%)
   

### WEMWBS: total sum score


In [25]:
# WEMWBS sum score and distribution

wemwbs_cols = [f'WEMWBS{i}' for i in range(1, 15)]  # WEMWBS1 .. WEMWBS14

# Build sum score (only when all 14 items are non-missing)
df['WEMWBS_sum'] = df[wemwbs_cols].apply(pd.to_numeric, errors='coerce').sum(axis=1)
mask_complete = df[wemwbs_cols].apply(pd.to_numeric, errors='coerce').notna().all(axis=1)
df.loc[~mask_complete, 'WEMWBS_sum'] = np.nan

wemwbs_sum = df['WEMWBS_sum'].dropna()
n_valid = len(wemwbs_sum)
n_miss = len(df) - n_valid

print("WEMWBS sum score (WEMWBS1 + … + WEMWBS14):")
print(f"  N with complete data: {n_valid:>6}")
print(f"  N missing (any item): {n_miss:>6}")
print()
if n_valid > 0:
    print("  Distribution:")
    print(f"    Mean (SD):   {wemwbs_sum.mean():.2f} ({wemwbs_sum.std():.2f})")
    print(f"    Median:     {wemwbs_sum.median():.2f}")
    print(f"    Min – Max:  {wemwbs_sum.min():.0f} – {wemwbs_sum.max():.0f}")
    q1, q3 = wemwbs_sum.quantile(0.25), wemwbs_sum.quantile(0.75)
    print(f"    IQR:        {q3 - q1:.2f}  (Q1 = {q1:.2f}, Q3 = {q3:.2f})")
    print()
    print("  Value counts:")
    for val, count in wemwbs_sum.value_counts().sort_index().items():
        pct = 100 * count / n_valid
        print(f"    {val:>3.0f}: {count:>6}  ({pct:.1f}%)")

WEMWBS sum score (WEMWBS1 + … + WEMWBS14):
  N with complete data:  12593
  N missing (any item):  16317

  Distribution:
    Mean (SD):   52.46 (8.37)
    Median:     54.00
    Min – Max:  14 – 70
    IQR:        11.00  (Q1 = 47.00, Q3 = 58.00)

  Value counts:
     14:      1  (0.0%)
     16:      1  (0.0%)
     19:      2  (0.0%)
     20:      4  (0.0%)
     21:      6  (0.0%)
     22:      8  (0.1%)
     23:      5  (0.0%)
     24:      6  (0.0%)
     25:      9  (0.1%)
     26:     14  (0.1%)
     27:     13  (0.1%)
     28:     31  (0.2%)
     29:     21  (0.2%)
     30:     23  (0.2%)
     31:     54  (0.4%)
     32:     44  (0.3%)
     33:     48  (0.4%)
     34:     78  (0.6%)
     35:     86  (0.7%)
     36:     97  (0.8%)
     37:    130  (1.0%)
     38:    162  (1.3%)
     39:    190  (1.5%)
     40:    184  (1.5%)
     41:    192  (1.5%)
     42:    219  (1.7%)
     43:    228  (1.8%)
     44:    304  (2.4%)
     45:    317  (2.5%)
     46:    313  (2.5%)
     47:    364  

### PHQ-4: total sum score


In [26]:
# PHQ-4 sum score and distribution (recode 1-4 -> 0-3)

phq4_cols = ['PHQ4_1', 'PHQ4_2', 'PHQ4_3', 'PHQ4_4']

# Convert to numeric
phq4_num = df[phq4_cols].apply(pd.to_numeric, errors='coerce')

# Recode: 1->0, 2->1, 3->2, 4->3 (anything else -> NaN)
phq4_recoded = phq4_num.replace({1: 0, 2: 1, 3: 2, 4: 3})
phq4_recoded = phq4_recoded.where(phq4_num.isin([1, 2, 3, 4]), np.nan)

# Optional: store recoded items in dataframe
for c in phq4_cols:
    df[c + '_recoded'] = phq4_recoded[c]

# Build sum score only when all 4 recoded items are non-missing
mask_complete = phq4_recoded.notna().all(axis=1)
df['PHQ4_sum'] = phq4_recoded.sum(axis=1)
df.loc[~mask_complete, 'PHQ4_sum'] = np.nan

# Summary output
phq4_sum = df['PHQ4_sum'].dropna()
n_valid = len(phq4_sum)
n_miss = len(df) - n_valid

print("PHQ-4 sum score (recoded: 1-4 -> 0-3):")
print(f"  N with complete data: {n_valid:>6}")
print(f"  N missing (any item): {n_miss:>6}")
print()

if n_valid > 0:
    print("  Distribution:")
    print(f"    Mean (SD):   {phq4_sum.mean():.2f} ({phq4_sum.std():.2f})")
    print(f"    Median:     {phq4_sum.median():.2f}")
    print(f"    Min - Max:  {phq4_sum.min():.0f} - {phq4_sum.max():.0f}")
    q1, q3 = phq4_sum.quantile(0.25), phq4_sum.quantile(0.75)
    print(f"    IQR:        {q3 - q1:.2f}  (Q1 = {q1:.2f}, Q3 = {q3:.2f})")
    print()
    print("  Value counts:")
    for val, count in phq4_sum.value_counts().sort_index().items():
        pct = 100 * count / n_valid
        print(f"    {val:>3.0f}: {count:>6}  ({pct:.1f}%)")

PHQ-4 sum score (recoded: 1-4 -> 0-3):
  N with complete data:  12843
  N missing (any item):  16067

  Distribution:
    Mean (SD):   2.06 (2.25)
    Median:     1.00
    Min - Max:  0 - 12
    IQR:        3.00  (Q1 = 0.00, Q3 = 3.00)

  Value counts:
      0:   3900  (30.4%)
      1:   2604  (20.3%)
      2:   2086  (16.2%)
      3:   1490  (11.6%)
      4:   1294  (10.1%)
      5:    485  (3.8%)
      6:    329  (2.6%)
      7:    196  (1.5%)
      8:    175  (1.4%)
      9:    100  (0.8%)
     10:     83  (0.6%)
     11:     42  (0.3%)
     12:     59  (0.5%)


### DSM-5 Cross-Cutting Symptom Measure (DSM-5 CCSM)


In [27]:
# DSM5CC sum score and distribution

dsm5cc_cols = ['DSM5CC_3', 'DSM5CC_6', 'DSM5CC_7', 'DSM5CC_8a', 'DSM5CC_10', 'DSM5CC_15', 'DSM5CC_9']
dsm5cc_cols = [c for c in dsm5cc_cols if c in df.columns]

# Build sum score (only when all items non-missing)
df['DSM5CC_sum'] = df[dsm5cc_cols].apply(pd.to_numeric, errors='coerce').sum(axis=1)
mask_complete = df[dsm5cc_cols].apply(pd.to_numeric, errors='coerce').notna().all(axis=1)
df.loc[~mask_complete, 'DSM5CC_sum'] = np.nan

dsm5cc_sum = df['DSM5CC_sum'].dropna()
n_valid = len(dsm5cc_sum)
n_miss = len(df) - n_valid

print("DSM5CC sum score (DSM5CC_3 + DSM5CC_6 + DSM5CC_7 + DSM5CC_8a + DSM5CC_10 + DSM5CC_15 + DSM5CC_9):")
print(f"  N with complete data: {n_valid:>6}")
print(f"  N missing (any item): {n_miss:>6}")
print()
if n_valid > 0:
    print("  Distribution:")
    print(f"    Mean (SD):   {dsm5cc_sum.mean():.2f} ({dsm5cc_sum.std():.2f})")
    print(f"    Median:     {dsm5cc_sum.median():.2f}")
    print(f"    Min – Max:  {dsm5cc_sum.min():.0f} – {dsm5cc_sum.max():.0f}")
    q1, q3 = dsm5cc_sum.quantile(0.25), dsm5cc_sum.quantile(0.75)
    print(f"    IQR:        {q3 - q1:.2f}  (Q1 = {q1:.2f}, Q3 = {q3:.2f})")
    print()
    print("  Value counts:")
    for val, count in dsm5cc_sum.value_counts().sort_index().items():
        pct = 100 * count / n_valid
        print(f"    {val:>3.0f}: {count:>6}  ({pct:.1f}%)")

DSM5CC sum score (DSM5CC_3 + DSM5CC_6 + DSM5CC_7 + DSM5CC_8a + DSM5CC_10 + DSM5CC_15 + DSM5CC_9):
  N with complete data:  12843
  N missing (any item):  16067

  Distribution:
    Mean (SD):   12.83 (4.49)
    Median:     12.00
    Min – Max:  7 – 35
    IQR:        5.00  (Q1 = 10.00, Q3 = 15.00)

  Value counts:
      7:    655  (5.1%)
      8:   1093  (8.5%)
      9:   1461  (11.4%)
     10:   1509  (11.7%)
     11:   1397  (10.9%)
     12:   1183  (9.2%)
     13:   1057  (8.2%)
     14:    833  (6.5%)
     15:    699  (5.4%)
     16:    563  (4.4%)
     17:    479  (3.7%)
     18:    407  (3.2%)
     19:    336  (2.6%)
     20:    284  (2.2%)
     21:    209  (1.6%)
     22:    158  (1.2%)
     23:    136  (1.1%)
     24:     90  (0.7%)
     25:     76  (0.6%)
     26:     56  (0.4%)
     27:     60  (0.5%)
     28:     32  (0.2%)
     29:     27  (0.2%)
     30:     14  (0.1%)
     31:     13  (0.1%)
     32:      6  (0.0%)
     33:      6  (0.0%)
     34:      3  (0.0%)
     35: 

## AI usage patterns


### Reasons for not using AI


In [28]:
# KI use: overview (Ja / Nein), then reasons for non-use among those who answered Nein

ki_use_col = 'ki_nutzen_sie_ki_programme_wie_z_b_chatgpt_gemini_claud'
ja_nein = ['Ja', 'ja', 'Nein', 'nein']

print("=" * 60)
print("1. KI use (ki_nutzen_sie_ki_programme_wie_z_b_chatgpt_gemini_claud)")
print("=" * 60)
use_counts = df[ki_use_col].astype(str).str.strip().value_counts(dropna=False)
use_pct = df[ki_use_col].astype(str).str.strip().replace('', np.nan).dropna().value_counts(normalize=True) * 100
for val, count in use_counts.items():
    if pd.isna(val) or str(val).strip() == '' or str(val) == 'nan':
        print(f"  (missing)  {count:>6}")
    else:
        pct = use_pct.get(val, 0.0)
        print(f"  {val:<10} {count:>6}  ({pct:.1f}%)")
print(f"  Total:     {len(df)}")
print()

# Subset: answered Nein to KI use
mask_nein = df[ki_use_col].astype(str).str.strip().str.lower() == 'nein'
df_no_ki = df.loc[mask_nein].copy()
n_no_ki = len(df_no_ki)
print("=" * 60)
print(f"2. Respondents who answered NEIN (N = {n_no_ki}) — reasons for not using AI")
print("=" * 60)

# Reason columns (each Ja/Nein)
reason_cols = [
    'ki_ich_sehe_keinen_konkreten_nutzen_fuer_meinen_alltag',   # No concrete benefit
    'ki_ich_habe_sorge_vor_falschinformationen',                # Concern about misinformation
    'ki_ich_moechte_keine_sensiblen_daten_oder_privaten_deta',  # Don't want sensitive data
    'ki_warum_nutzen_sie_keine_ki_programme',                   # Why don't you use (context)
    'ki_ich_weiss_nicht_wie_ich_ki_fuer_spezielle_fragen_sin',  # Don't know how for specific questions
    'ki_ich_lehne_den_einsatz_von_ki_aus_prinzipiellen_ethis',  # Reject AI on principle
    'ki_sonstiges',                                              # Other
]

reason_labels = {
    'ki_ich_sehe_keinen_konkreten_nutzen_fuer_meinen_alltag': 'No concrete benefit',
    'ki_ich_habe_sorge_vor_falschinformationen': 'Concern about misinformation',
    'ki_ich_moechte_keine_sensiblen_daten_oder_privaten_deta': "Don't want sensitive/private data",
    'ki_warum_nutzen_sie_keine_ki_programme': 'I do not know who is responsible if something goes wrong',
    'ki_ich_weiss_nicht_wie_ich_ki_fuer_spezielle_fragen_sin': "Don't know how for specific questions",
    'ki_ich_lehne_den_einsatz_von_ki_aus_prinzipiellen_ethis': 'Reject AI on principle',
    'ki_sonstiges': 'Other (sonstiges)',
}

for col in reason_cols:
    if col not in df_no_ki.columns:
        print(f"\n  [{col}] — column not found, skipped")
        continue
    s = df_no_ki[col].astype(str).str.strip()
    counts = s.value_counts(dropna=False)
    valid = s.replace('', np.nan).replace('nan', np.nan).dropna()
    pct_series = valid.value_counts(normalize=True) * 100 if len(valid) > 0 else pd.Series(dtype=float)
    label = reason_labels.get(col, col)
    print(f"\n  {label}")
    print(f"  {'Value':<12} {'N':>8}  {'%':>8}")
    print("  " + "-" * 30)
    for val, count in counts.items():
        if pd.isna(val) or str(val).strip() == '' or str(val) == 'nan':
            print(f"  {'(missing)':<12} {count:>8}")
        else:
            pct = pct_series.get(val, 0.0)
            print(f"  {val:<12} {count:>8}  {pct:>7.1f}%")
    print(f"  {'Total':<12} {len(df_no_ki):>8}")

# Free-text: other reasons (only for those who answered Ja to ki_sonstiges)
text_col = 'ki_was_sind_weitere_gruende_weshalb_sie_keine_ki_progra'
print("\n" + "=" * 60)
print("3. Free-text: other reasons (among those who answered Ja to ki_sonstiges)")
print("=" * 60)
if text_col not in df_no_ki.columns:
    print(f"  Column '{text_col}' not found.")
else:
    mask_sonstiges_ja = df_no_ki['ki_sonstiges'].astype(str).str.strip().str.lower() == 'ja'
    df_other = df_no_ki.loc[mask_sonstiges_ja, text_col]
    n_other_ja = mask_sonstiges_ja.sum()
    # Non-empty text
    text_vals = df_other.astype(str).str.strip()
    text_filled = text_vals.replace('', np.nan).replace('nan', np.nan).dropna()
    n_with_text = len(text_filled)
    print(f"  N answered Ja to ki_sonstiges: {n_other_ja}")
    print(f"  N with non-empty text:         {n_with_text}")
    if n_with_text > 0:
        print("\n  Value counts (sample of unique answers, max 50):")
        vc = text_filled.value_counts()
        for i, (val, count) in enumerate(vc.items()):
            if i >= 50:
                print(f"  ... and {len(vc) - 50} more unique answers")
                break
            snippet = (val[:70] + '...') if len(val) > 70 else val
            print(f"    n={count:>5}  {snippet}")
    else:
        print("  (No non-empty free-text answers.)")

1. KI use (ki_nutzen_sie_ki_programme_wie_z_b_chatgpt_gemini_claud)
  Ja          15058  (52.1%)
  Nein        13852  (47.9%)
  Total:     28910

2. Respondents who answered NEIN (N = 13852) — reasons for not using AI

  No concrete benefit
  Value               N         %
  ------------------------------
  Ja               8953     64.7%
  Nein             4876     35.3%
  (missing)          23
  Total           13852

  Concern about misinformation
  Value               N         %
  ------------------------------
  Nein             8313     60.1%
  Ja               5516     39.9%
  (missing)          23
  Total           13852

  Don't want sensitive/private data
  Value               N         %
  ------------------------------
  Nein             6944     50.2%
  Ja               6885     49.8%
  (missing)          23
  Total           13852

  I do not know who is responsible if something goes wrong
  Value               N         %
  ------------------------------
  Nein        

### AI tools used


In [29]:
# AI tools used (among those who said Ja to using AI) + missing counts per tool

import numpy as np
import pandas as pd

ki_use_col = "ki_nutzen_sie_ki_programme_wie_z_b_chatgpt_gemini_claud"
mask_use_ki = df[ki_use_col].astype(str).str.strip().str.lower() == "ja"
df_use_ki = df.loc[mask_use_ki].copy()
n_use_ki = len(df_use_ki)

tool_cols = [
    "ki_chatgpt_von_openai",
    "ki_gemini_von_google",
    "ki_claude_von_anthropic",
    "ki_grok_von_x",
    "ki_perplexity_ai",
    "ki_llama_von_meta",
    "ki_deepseek_von_deepseek",
    "ki_mistral_von_mistral_ai",
    "ki_copilot_von_microsoft",
]
tool_labels = {
    "ki_chatgpt_von_openai": "ChatGPT (OpenAI)",
    "ki_gemini_von_google": "Gemini (Google)",
    "ki_claude_von_anthropic": "Claude (Anthropic)",
    "ki_grok_von_x": "Grok (X)",
    "ki_perplexity_ai": "Perplexity AI",
    "ki_llama_von_meta": "Llama (Meta)",
    "ki_deepseek_von_deepseek": "DeepSeek",
    "ki_mistral_von_mistral_ai": "Mistral (Mistral AI)",
    "ki_copilot_von_microsoft": "Copilot (Microsoft)",
    "ki_sonstige": "Other (ki_sonstige)",
}

all_tool_cols = tool_cols + ["ki_sonstige"]

def is_missing_tool(x) -> bool:
    # Treat true NaN + empty string + literal "nan" (from astype(str)) as missing
    if pd.isna(x):
        return True
    s = str(x).strip().lower()
    return (s == "") or (s == "nan")

print("=" * 70)
print(f"Respondents who said they USE AI (Ja): N = {n_use_ki}")
print("=" * 70)
print("\nTool usage (Ja/Nein) in this subgroup (denominator = all AI users):\n")
print(f"  {'Tool':<28} {'Ja':>8} {'Nein':>8} {'Missing':>9}  % Ja")
print("  " + "-" * 62)

for col in all_tool_cols:
    label = tool_labels.get(col, col)

    if col not in df_use_ki.columns:
        print(f"  {label:<28} (column missing)")
        continue

    s_raw = df_use_ki[col]

    # Missing among AI users
    miss = s_raw.apply(is_missing_tool).sum()

    # Ja/Nein counts (only exact Ja/Nein after stripping; everything else is neither)
    s = s_raw.astype(str).str.strip().str.lower()
    ja = (s == "ja").sum()
    nein = (s == "nein").sum()

    pct_ja = 100 * ja / n_use_ki if n_use_ki else 0.0
    print(f"  {label:<28} {ja:>8} {nein:>8} {miss:>9}  {pct_ja:>5.1f}%")

print("\nMissingness summary (among AI users):")
for col in all_tool_cols:
    if col not in df_use_ki.columns:
        continue
    miss = df_use_ki[col].apply(is_missing_tool).sum()
    print(f"  {tool_labels.get(col, col):<28}: missing {miss:>6} / {n_use_ki} ({(100*miss/n_use_ki if n_use_ki else 0):.1f}%)")

Respondents who said they USE AI (Ja): N = 15058

Tool usage (Ja/Nein) in this subgroup (denominator = all AI users):

  Tool                               Ja     Nein   Missing  % Ja
  --------------------------------------------------------------
  ChatGPT (OpenAI)                11077     3978         3   73.6%
  Gemini (Google)                  6346     8709         3   42.1%
  Claude (Anthropic)                517    14538         3    3.4%
  Grok (X)                          257    14798         3    1.7%
  Perplexity AI                    1474    13581         3    9.8%
  Llama (Meta)                      199    14856         3    1.3%
  DeepSeek                          298    14757         3    2.0%
  Mistral (Mistral AI)              334    14721         3    2.2%
  Copilot (Microsoft)              3315    11740         3   22.0%
  Other (ki_sonstige)              1616    13439         3   10.7%

Missingness summary (among AI users):
  ChatGPT (OpenAI)            : missing   

## Health-related AI use


### Specific use cases


In [30]:

ki_use_col = 'ki_nutzen_sie_ki_programme_wie_z_b_chatgpt_gemini_claud'
mask_use_ki = df[ki_use_col].astype(str).str.strip().str.lower() == 'ja'
df_use_ki = df.loc[mask_use_ki].copy()
n_use_ki = len(df_use_ki)

# Categorical purpose variables (G1Q4.1-G1Q4.13)
purpose_cols = [
    'ki_wie_haeufig_nutzen_sie_ki_programme_fuer_die_folgend',
    'ki_wie_haeufig_nutzen_sie_ki_programme_fuer_die_folgend_2',
    'ki_wie_haeufig_nutzen_sie_ki_programme_fuer_die_folgend_3',
    'ki_bei_aengsten_und_sorgen',
    'ki_bei_selbstzweifeln_oder_niedergeschlagenheit',
    'ki_bei_einsamkeit_z_b_um_jemanden_zum_reden_zu_haben_we',
    'ki_bei_zwischenmenschlichen_konflikten',
    'ki_bei_unsicherheiten_bei_entscheidungen_um_meine_gedan',
    'ki_bei_trauer_oder_verlust',
    'ki_wenn_grosse_lebensveraenderungen_anstehen',
    'ki_wie_haeufig_nutzen_sie_ki_programme_fuer_die_folgend_4',
    'ki_wie_haeufig_nutzen_sie_ki_programme_fuer_die_folgend_5',
    'ki_fuer_sonstige_gesundheits_psychische_oder_emotionale',
]
purpose_labels = {
    'ki_wie_haeufig_nutzen_sie_ki_programme_fuer_die_folgend': 'G1Q4.1: Symptom check and diagnostic help',
    'ki_wie_haeufig_nutzen_sie_ki_programme_fuer_die_folgend_2': 'G1Q4.2: Understanding medical documents',
    'ki_wie_haeufig_nutzen_sie_ki_programme_fuer_die_folgend_3': 'G1Q4.3: Fitness, nutrition & prevention',
    'ki_bei_aengsten_und_sorgen': 'G1Q4.4: For anxieties and worries',
    'ki_bei_selbstzweifeln_oder_niedergeschlagenheit': 'G1Q4.5: For self-doubt or dejection',
    'ki_bei_einsamkeit_z_b_um_jemanden_zum_reden_zu_haben_we': 'G1Q4.6: For loneliness (e.g. someone to talk to)',
    'ki_bei_zwischenmenschlichen_konflikten': 'G1Q4.7: For interpersonal conflicts',
    'ki_bei_unsicherheiten_bei_entscheidungen_um_meine_gedan': 'G1Q4.8: For uncertainties in decisions / organizing thoughts',
    'ki_bei_trauer_oder_verlust': 'G1Q4.9: For grief or loss',
    'ki_wenn_grosse_lebensveraenderungen_anstehen': 'G1Q4.10: When major life changes are impending',
    'ki_wie_haeufig_nutzen_sie_ki_programme_fuer_die_folgend_4': 'G1Q4.11: When acting impulsively or consuming more than good',
    'ki_wie_haeufig_nutzen_sie_ki_programme_fuer_die_folgend_5': 'G1Q4.12: In acute crises (overwhelmed, threatened, unable to act)',
    'ki_fuer_sonstige_gesundheits_psychische_oder_emotionale': 'G1Q4.13: For other health/psychological/emotional topics',
}

print("=" * 70)
print(f"AI use for specific purposes — among those who USE AI (N = {n_use_ki})")
print("=" * 70)

for col in purpose_cols:
    if col not in df_use_ki.columns:
        print(f"\n[{purpose_labels.get(col, col)}] — column not found, skipped")
        continue
    s = df_use_ki[col].astype(str).str.strip()
    counts = s.value_counts(dropna=False)
    valid = s.replace('', np.nan).replace('nan', np.nan).dropna()
    n_valid = len(valid)
    pct_series = (valid.value_counts(normalize=True) * 100) if n_valid > 0 else pd.Series(dtype=float)
    label = purpose_labels.get(col, col)
    print(f"\n  {label}")
    print(f"  {'Answer':<50} {'N':>8}  {'%':>7}")
    print("  " + "-" * 68)
    for val, count in counts.items():
        if pd.isna(val) or str(val).strip() == '' or str(val) == 'nan':
            print(f"  {'(missing)':<50} {count:>8}")
        else:
            pct = pct_series.get(val, 0.0)
            snippet = (val[:48] + '..') if len(val) > 50 else val
            print(f"  {snippet:<50} {count:>8}  {pct:>6.1f}%")
    print(f"  {'Total':<50} {len(df_use_ki):>8}")

# Summarize AI use for "grief or loss" and "major life changes" combined (unique people who use AI for either)
col_grief_loss = 'ki_bei_trauer_oder_verlust'
col_life_changes = 'ki_wenn_grosse_lebensveraenderungen_anstehen'

# Check that columns exist
cols_present = [col for col in [col_grief_loss, col_life_changes] if col in df_use_ki.columns]

def is_used(val):
    if pd.isna(val) or str(val).strip() == '' or str(val).strip().lower() == 'nan':
        return False
    v = str(val).strip().lower()
    return v not in ['hierfür nutze ich keine ki', '(bisher) kein bedarf, würde ki aber dafür nutzen']

if len(cols_present) == 2:
    used_grief = df_use_ki[col_grief_loss].apply(is_used)
    used_life_changes = df_use_ki[col_life_changes].apply(is_used)
    used_either = used_grief | used_life_changes
    n_used_either = used_either.sum()
    print("\n" + "=" * 70)
    print(f"AI use for grief/loss OR life changes (N = {n_use_ki})")
    print("=" * 70)
    print(f"Number of unique people who use AI for EITHER grief/loss or life changes: {n_used_either} / {n_use_ki}  ({n_used_either/n_use_ki*100:.1f}%)")
else:
    print("\nOne or both of the expected columns for 'grief/loss' or 'life changes' are missing, summary skipped.")

AI use for specific purposes — among those who USE AI (N = 15058)

  G1Q4.1: Symptom check and diagnostic help
  Answer                                                    N        %
  --------------------------------------------------------------------
  Hierfür nutze ich keine KI                             6349    42.2%
  Selten                                                 2680    17.8%
  Manchmal                                               2450    16.3%
  (Bisher) kein Bedarf, würde KI aber dafür nutzen       2329    15.5%
  Oft                                                     998     6.6%
  Immer                                                   240     1.6%
  (missing)                                                12
  Total                                                 15058

  G1Q4.2: Understanding medical documents
  Answer                                                    N        %
  --------------------------------------------------------------------
  Hierfür nu

### Proportion of AI users with health-related use (general health and mental health)


In [31]:
# Rate of AI users for medical use (all 13 purpose questions)
# Person-level: Yes / No / All missing

ki_use_col = 'ki_nutzen_sie_ki_programme_wie_z_b_chatgpt_gemini_claud'
mask_use_ki = df[ki_use_col].astype(str).str.strip().str.lower() == 'ja'
df_use_ki = df.loc[mask_use_ki].copy()
n_use_ki = len(df_use_ki)

purpose_cols_13 = [
    'ki_wie_haeufig_nutzen_sie_ki_programme_fuer_die_folgend',
    'ki_wie_haeufig_nutzen_sie_ki_programme_fuer_die_folgend_2',
    'ki_wie_haeufig_nutzen_sie_ki_programme_fuer_die_folgend_3',
    'ki_bei_aengsten_und_sorgen',
    'ki_bei_selbstzweifeln_oder_niedergeschlagenheit',
    'ki_bei_einsamkeit_z_b_um_jemanden_zum_reden_zu_haben_we',
    'ki_bei_zwischenmenschlichen_konflikten',
    'ki_bei_unsicherheiten_bei_entscheidungen_um_meine_gedan',
    'ki_bei_trauer_oder_verlust',
    'ki_wenn_grosse_lebensveraenderungen_anstehen',
    'ki_wie_haeufig_nutzen_sie_ki_programme_fuer_die_folgend_4',
    'ki_wie_haeufig_nutzen_sie_ki_programme_fuer_die_folgend_5',
    'ki_fuer_sonstige_gesundheits_psychische_oder_emotionale',
]
purpose_cols_13 = [c for c in purpose_cols_13 if c in df_use_ki.columns]

non_medical_answers = [
    'hierfür nutze ich keine ki',
    '(bisher) kein bedarf, würde ki aber dafür nutzen',
]

use_label = []
for _, row in df_use_ki[purpose_cols_13].iterrows():
    vals = []
    for c in purpose_cols_13:
        v = row[c]
        if pd.isna(v) or str(v).strip() == '' or str(v).strip().lower() == 'nan':
            continue
        vals.append(str(v).strip().lower())

    if len(vals) == 0:
        use_label.append('All missing')
    elif any(v not in non_medical_answers for v in vals):
        use_label.append('Yes')
    else:
        use_label.append('No')

counts = pd.Series(use_label).value_counts()

print("Among those who said they USE AI (N = {}):".format(n_use_ki))
print("  (Based on the 13 purpose questions.)")
print()
for label in ['Yes', 'No', 'All missing']:
    n = counts.get(label, 0)
    pct = 100 * n / n_use_ki if n_use_ki > 0 else 0
    print("  {:<15} {:>6}  ({:.1f}%)".format(label, n, pct))
print("  {:<15} {:>6}".format('Total', n_use_ki))

Among those who said they USE AI (N = 15058):
  (Based on the 13 purpose questions.)

  Yes               9507  (63.1%)
  No                5539  (36.8%)
  All missing         12  (0.1%)
  Total            15058


### Proportion of AI users with mental health use only


In [32]:
# Rate of AI users for psychological/emotional support
# Person-level: Yes / No / All missing
# - once EXCLUDING "Sonstiges" (Q4.4–Q4.12)
# - once INCLUDING "Sonstiges" (Q4.4–Q4.13)

import pandas as pd
import numpy as np

ki_use_col = 'ki_nutzen_sie_ki_programme_wie_z_b_chatgpt_gemini_claud'
mask_use_ki = df[ki_use_col].astype(str).str.strip().str.lower() == 'ja'
df_use_ki = df.loc[mask_use_ki].copy()
n_use_ki = len(df_use_ki)

# Q4.4–Q4.12 (without "Sonstiges")
emo_cols_4_12 = [
    'ki_bei_aengsten_und_sorgen',
    'ki_bei_selbstzweifeln_oder_niedergeschlagenheit',
    'ki_bei_einsamkeit_z_b_um_jemanden_zum_reden_zu_haben_we',
    'ki_bei_zwischenmenschlichen_konflikten',
    'ki_bei_unsicherheiten_bei_entscheidungen_um_meine_gedan',
    'ki_bei_trauer_oder_verlust',
    'ki_wenn_grosse_lebensveraenderungen_anstehen',
    'ki_wie_haeufig_nutzen_sie_ki_programme_fuer_die_folgend_4',
    'ki_wie_haeufig_nutzen_sie_ki_programme_fuer_die_folgend_5',
]

# Q4.13 ("Sonstiges")
sonstige_col = 'ki_fuer_sonstige_gesundheits_psychische_oder_emotionale'

non_use_answers = {
    'hierfür nutze ich keine ki',
    '(bisher) kein bedarf, würde ki aber dafür nutzen',
}

def classify_person(row, cols):
    vals = []
    for c in cols:
        if c not in row.index:
            continue
        v = row[c]
        if pd.isna(v):
            continue
        s = str(v).strip().lower()
        if s in ['', 'nan']:
            continue
        vals.append(s)

    if len(vals) == 0:
        return 'All missing'
    elif any(v not in non_use_answers for v in vals):
        return 'Yes'
    else:
        return 'No'

def summarize_variant(label, cols):
    cols_present = [c for c in cols if c in df_use_ki.columns]
    if len(cols_present) == 0:
        print(f"\n{label}: keine der erwarteten Spalten gefunden.")
        return None

    use_label = [classify_person(row, cols_present) for _, row in df_use_ki[cols_present].iterrows()]
    counts = pd.Series(use_label).value_counts()

    print("\n" + "=" * 70)
    print(f"{label} (N AI users = {n_use_ki})")
    print("=" * 70)
    for cat in ['Yes', 'No', 'All missing']:
        n = counts.get(cat, 0)
        pct = 100 * n / n_use_ki if n_use_ki > 0 else 0
        print(f"  {cat:<15} {n:>6}  ({pct:.1f}%)")
    print(f"  {'Total':<15} {n_use_ki:>6}")

    return pd.Series(use_label, index=df_use_ki.index)

# Variante A: EXKLUSIVE Sonstiges (Q4.4–Q4.12)
labels_excl = summarize_variant(
    "Psych/Emotional use EXCLUDING Sonstiges (Q4.4–Q4.12)",
    emo_cols_4_12
)

# Variante B: INKLUSIVE Sonstiges (Q4.4–Q4.13)
emo_cols_4_13 = emo_cols_4_12 + [sonstige_col]
labels_incl = summarize_variant(
    "Psych/Emotional use INCLUDING Sonstiges (Q4.4–Q4.13)",
    emo_cols_4_13
)

# Zusatz: Wie viele werden NUR durch Sonstiges zu "Yes"?
if labels_excl is not None and labels_incl is not None:
    only_due_to_sonstige = ((labels_excl != 'Yes') & (labels_incl == 'Yes')).sum()
    pct_only_due = 100 * only_due_to_sonstige / n_use_ki if n_use_ki > 0 else 0

    print("\n" + "-" * 70)
    print(f"Nur wegen Sonstiges zusätzlich als 'Yes': {only_due_to_sonstige} ({pct_only_due:.1f}%)")
    print("-" * 70)


Psych/Emotional use EXCLUDING Sonstiges (Q4.4–Q4.12) (N AI users = 15058)
  Yes               4312  (28.6%)
  No               10733  (71.3%)
  All missing         13  (0.1%)
  Total            15058

Psych/Emotional use INCLUDING Sonstiges (Q4.4–Q4.13) (N AI users = 15058)
  Yes               5097  (33.8%)
  No                9948  (66.1%)
  All missing         13  (0.1%)
  Total            15058

----------------------------------------------------------------------
Nur wegen Sonstiges zusätzlich als 'Yes': 785 (5.2%)
----------------------------------------------------------------------


### Proportion of AI users with general health use only


In [33]:
# Person-level medical use classification (Q1–Q3, among AI users)

ki_use_col = 'ki_nutzen_sie_ki_programme_wie_z_b_chatgpt_gemini_claud'
mask_use_ki = df[ki_use_col].astype(str).str.strip().str.lower() == 'ja'
df_use_ki = df.loc[mask_use_ki].copy()
n_use_ki = len(df_use_ki)

purpose_cols_1_3 = [
    'ki_wie_haeufig_nutzen_sie_ki_programme_fuer_die_folgend',
    'ki_wie_haeufig_nutzen_sie_ki_programme_fuer_die_folgend_2',
    'ki_wie_haeufig_nutzen_sie_ki_programme_fuer_die_folgend_3',
]
purpose_cols_1_3 = [c for c in purpose_cols_1_3 if c in df_use_ki.columns]

non_use_answers = [
    'hierfür nutze ich keine ki',
    '(bisher) kein bedarf, würde ki aber dafür nutzen',
]

med_use_label = []
for _, row in df_use_ki[purpose_cols_1_3].iterrows():
    vals = []
    for c in purpose_cols_1_3:
        v = row[c]
        if pd.isna(v) or str(v).strip() == '' or str(v).strip().lower() == 'nan':
            continue
        vals.append(str(v).strip().lower())

    if len(vals) == 0:
        med_use_label.append('All missing')
    elif any(v not in non_use_answers for v in vals):
        med_use_label.append('Yes')
    else:
        med_use_label.append('No')

counts = pd.Series(med_use_label).value_counts()
print(f"Medical use classification (Q1–Q3, among AI users, N = {n_use_ki}):\n")
for label in ['Yes', 'No', 'All missing']:
    n = counts.get(label, 0)
    pct = 100 * n / n_use_ki if n_use_ki > 0 else 0
    print(f"  {label:<15} {n:>6}  ({pct:.1f}%)")
print(f"  {'Total':<15} {n_use_ki:>6}")

Medical use classification (Q1–Q3, among AI users, N = 15058):

  Yes               8727  (58.0%)
  No                6319  (42.0%)
  All missing         12  (0.1%)
  Total            15058


### Reasons for not using AI for health-related purposes

Among participants who use AI in general, this section summarizes reasons for **not** using AI for health-related purposes


In [34]:
# Reasons for not using AI for medical/health/psychological/emotional purposes
# (among those who use AI but do NOT use it for any of the 13 G1Q4 topics)
# Variables: G1Q8.1–G1Q8.7, G1Q9 — with readable labels from the survey

ki_use_col = 'ki_nutzen_sie_ki_programme_wie_z_b_chatgpt_gemini_claud'
mask_use_ki = df[ki_use_col].astype(str).str.strip().str.lower() == 'ja'
df_use_ki = df.loc[mask_use_ki].copy()
n_use_ki = len(df_use_ki)

# G1Q4.1–G1Q4.13: all medical + psychological/emotional purposes
purpose_cols_1_13 = [
    'ki_wie_haeufig_nutzen_sie_ki_programme_fuer_die_folgend',      # G1Q4.1  Symptom check / diagnosis help
    'ki_wie_haeufig_nutzen_sie_ki_programme_fuer_die_folgend_2',    # G1Q4.2  Understanding medical documents
    'ki_wie_haeufig_nutzen_sie_ki_programme_fuer_die_folgend_3',    # G1Q4.3  Fitness, nutrition & prevention
    'ki_bei_aengsten_und_sorgen',                                   # G1Q4.4
    'ki_bei_selbstzweifeln_oder_niedergeschlagenheit',              # G1Q4.5
    'ki_bei_einsamkeit_z_b_um_jemanden_zum_reden_zu_haben_we',      # G1Q4.6
    'ki_bei_zwischenmenschlichen_konflikten',                        # G1Q4.7
    'ki_bei_unsicherheiten_bei_entscheidungen_um_meine_gedan',      # G1Q4.8
    'ki_bei_trauer_oder_verlust',                                   # G1Q4.9
    'ki_wenn_grosse_lebensveraenderungen_anstehen',                  # G1Q4.10
    'ki_wie_haeufig_nutzen_sie_ki_programme_fuer_die_folgend_4',    # G1Q4.11
    'ki_wie_haeufig_nutzen_sie_ki_programme_fuer_die_folgend_5',    # G1Q4.12
    'ki_fuer_sonstige_gesundheits_psychische_oder_emotionale',      # G1Q4.13
]
purpose_cols_1_13 = [c for c in purpose_cols_1_13 if c in df_use_ki.columns]

non_use_answers = [
    'hierfür nutze ich keine ki',
    '(bisher) kein bedarf, würde ki aber dafür nutzen',
]

def is_non_use_only(val):
    if pd.isna(val) or str(val).strip() == '' or str(val).strip().lower() == 'nan':
        return True
    return str(val).strip().lower() in non_use_answers

# Non‑use across ALL 13 purposes
all_non_use_1_13 = pd.Series(True, index=df_use_ki.index)
for col in purpose_cols_1_13:
    all_non_use_1_13 &= df_use_ki[col].apply(is_non_use_only)

df_no_medical_psych = df_use_ki.loc[all_non_use_1_13].copy()
n_no_medical_psych = len(df_no_medical_psych)

# Among this non‑use group, how many were actually presented with G1Q8/G1Q9?
reason_cols_existing = [c for c in [
    'ki_ich_sehe_keinen_konkreten_nutzen_fuer_meinen_alltag_2',
    'ki_ich_habe_sorge_vor_falschinformationen_2',
    'ki_weshalb_nutzen_sie_keine_ki_programme_fuer_gesundhei',
    'ki_weshalb_nutzen_sie_keine_ki_programme_fuer_gesundhei_2',
    'ki_weshalb_nutzen_sie_keine_ki_programme_fuer_gesundhei_3',
    'ki_weshalb_nutzen_sie_keine_ki_programme_fuer_gesundhei_4',
    'ki_sonstiges_2',
] if c in df_no_medical_psych.columns]
if reason_cols_existing:
    tmp = (
        df_no_medical_psych[reason_cols_existing]
        .astype(str)
        .apply(lambda s: s.str.strip())
        .replace({'': np.nan, 'nan': np.nan})
    )
    mask_presented = tmp.notna().any(axis=1)
    n_presented_reasons = mask_presented.sum()
else:
    n_presented_reasons = 0

# G1Q8 / G1Q9: reasons for not using AI for health (readable labels)
reason_cols_medical = [
    'ki_ich_sehe_keinen_konkreten_nutzen_fuer_meinen_alltag_2',
    'ki_ich_habe_sorge_vor_falschinformationen_2',
    'ki_weshalb_nutzen_sie_keine_ki_programme_fuer_gesundhei',
    'ki_weshalb_nutzen_sie_keine_ki_programme_fuer_gesundhei_2',
    'ki_weshalb_nutzen_sie_keine_ki_programme_fuer_gesundhei_3',
    'ki_weshalb_nutzen_sie_keine_ki_programme_fuer_gesundhei_4',
    'ki_sonstiges_2',
]
reason_labels_medical = {
    'ki_ich_sehe_keinen_konkreten_nutzen_fuer_meinen_alltag_2': 'G1Q8.1 — I see no concrete benefit for my daily life',
    'ki_ich_habe_sorge_vor_falschinformationen_2': 'G1Q8.2 — I am concerned about misinformation',
    'ki_weshalb_nutzen_sie_keine_ki_programme_fuer_gesundhei': "G1Q8.3 — I don't want to share sensitive health data or private details with an AI",
    'ki_weshalb_nutzen_sie_keine_ki_programme_fuer_gesundhei_2': "G1Q8.4 — I don't know who is responsible if the AI gives me wrong advice",
    'ki_weshalb_nutzen_sie_keine_ki_programme_fuer_gesundhei_3': "G1Q8.5 — I don't know how I can meaningfully use AI specifically for health questions",
    'ki_weshalb_nutzen_sie_keine_ki_programme_fuer_gesundhei_4': 'G1Q8.6 — I reject the use of AI in health topics for principled/ethical reasons',
    'ki_sonstiges_2': 'G1Q8.7 — Other (Sonstiges)',
}

print('=' * 72)
print('Reasons for not using AI for medical/health/psychological/emotional purposes — among those who USE AI')
print(f'  and report NO use for any of the 13 G1Q4 purposes; N = {n_no_medical_psych}')
print(f'  of these, N actually presented with G1Q8/G1Q9 reasons screen: {n_presented_reasons}')
print('=' * 72)

for col in reason_cols_medical:
    if col not in df_no_medical_psych.columns:
        print(f"\n  [{reason_labels_medical.get(col, col)}] — column not found, skipped")
        continue
    s = df_no_medical_psych[col].astype(str).str.strip()
    counts = s.value_counts(dropna=False)
    valid = s.replace('', np.nan).replace('nan', np.nan).dropna()
    pct_series = valid.value_counts(normalize=True) * 100 if len(valid) > 0 else pd.Series(dtype=float)
    label = reason_labels_medical.get(col, col)
    print(f'\n  {label}')
    print("  {'Value':<12} {'N':>8}  {'%':>8}")
    print('  ' + '-' * 30)
    for val, count in counts.items():
        if pd.isna(val) or str(val).strip() == '' or str(val) == 'nan':
            print(f"  {'(missing)':<12} {count:>8}")
        else:
            pct = pct_series.get(val, 0.0)
            print(f'  {val:<12} {count:>8}  {pct:>7.1f}%')
    print(f"  {'Total':<12} {len(df_no_medical_psych):>8}")

# Free-text G1Q9
text_col_medical = 'ki_was_sind_weitere_gruende_weshalb_sie_keine_ki_progra_2'
print('\n' + '=' * 72)
print('G1Q9 — Free-text: other reasons (among those who answered Ja to G1Q8.7 Sonstiges)')
print('=' * 72)
if text_col_medical not in df_no_medical_psych.columns:
    print(f"  Column '{text_col_medical}' not found.")
else:
    mask_sonstiges_ja = df_no_medical_psych['ki_sonstiges_2'].astype(str).str.strip().str.lower() == 'ja'
    df_other_med = df_no_medical_psych.loc[mask_sonstiges_ja, text_col_medical]
    n_other_ja = mask_sonstiges_ja.sum()
    text_vals = df_other_med.astype(str).str.strip()
    text_filled = text_vals.replace('', np.nan).replace('nan', np.nan).dropna()
    n_with_text = len(text_filled)
    print(f'  N answered Ja to G1Q8.7 (Sonstiges): {n_other_ja}')
    print(f'  N with non-empty text:                {n_with_text}')
    if n_with_text > 0:
        print('\n  Value counts (sample of unique answers, max 50):')
        vc = text_filled.value_counts()
        for i, (val, count) in enumerate(vc.items()):
            if i >= 50:
                print(f'  ... and {len(vc) - 50} more unique answers')
                break
            snippet = (val[:70] + '...') if len(val) > 70 else val
            print(f'    n={count:>5}  {snippet}')
    else:
        print('  (No non-empty free-text answers.)')

Reasons for not using AI for medical/health/psychological/emotional purposes — among those who USE AI
  and report NO use for any of the 13 G1Q4 purposes; N = 5551
  of these, N actually presented with G1Q8/G1Q9 reasons screen: 3177

  G1Q8.1 — I see no concrete benefit for my daily life
  {'Value':<12} {'N':>8}  {'%':>8}
  ------------------------------
  (missing)        2374
  Ja               1791     56.4%
  Nein             1386     43.6%
  Total            5551

  G1Q8.2 — I am concerned about misinformation
  {'Value':<12} {'N':>8}  {'%':>8}
  ------------------------------
  (missing)        2374
  Ja               1803     56.8%
  Nein             1374     43.2%
  Total            5551

  G1Q8.3 — I don't want to share sensitive health data or private details with an AI
  {'Value':<12} {'N':>8}  {'%':>8}
  ------------------------------
  (missing)        2374
  Ja               1787     56.2%
  Nein             1390     43.8%
  Total            5551

  G1Q8.4 — I don't know 

## Downstream consequences of health-related AI use


### Downstream behaviors and emotional reactions

Assessed among AI users who report health-related use.


In [35]:
# Consequences of AI usage (G2Q1.1–G2Q1.7)
# Among AI users who actually use AI for at least one of the 13 health/psychological/emotional purposes (G1Q4.1–G1Q4.13)
# Question stem: "Has the interaction with an AI program for health, psychological or emotional topics ever led to you …"

ki_use_col = 'ki_nutzen_sie_ki_programme_wie_z_b_chatgpt_gemini_claud'
mask_use_ki = df[ki_use_col].astype(str).str.strip().str.lower() == 'ja'
df_use_ki = df.loc[mask_use_ki].copy()
n_use_ki = len(df_use_ki)

# G1Q4.1–G1Q4.13: all medical + psychological/emotional purposes
purpose_cols_1_13 = [
    'ki_wie_haeufig_nutzen_sie_ki_programme_fuer_die_folgend',      # G1Q4.1
    'ki_wie_haeufig_nutzen_sie_ki_programme_fuer_die_folgend_2',    # G1Q4.2
    'ki_wie_haeufig_nutzen_sie_ki_programme_fuer_die_folgend_3',    # G1Q4.3
    'ki_bei_aengsten_und_sorgen',                                   # G1Q4.4
    'ki_bei_selbstzweifeln_oder_niedergeschlagenheit',              # G1Q4.5
    'ki_bei_einsamkeit_z_b_um_jemanden_zum_reden_zu_haben_we',      # G1Q4.6
    'ki_bei_zwischenmenschlichen_konflikten',                        # G1Q4.7
    'ki_bei_unsicherheiten_bei_entscheidungen_um_meine_gedan',      # G1Q4.8
    'ki_bei_trauer_oder_verlust',                                   # G1Q4.9
    'ki_wenn_grosse_lebensveraenderungen_anstehen',                  # G1Q4.10
    'ki_wie_haeufig_nutzen_sie_ki_programme_fuer_die_folgend_4',    # G1Q4.11
    'ki_wie_haeufig_nutzen_sie_ki_programme_fuer_die_folgend_5',    # G1Q4.12
    'ki_fuer_sonstige_gesundheits_psychische_oder_emotionale',      # G1Q4.13
]
purpose_cols_1_13 = [c for c in purpose_cols_1_13 if c in df_use_ki.columns]

non_use_answers = [
    'hierfür nutze ich keine ki',
    '(bisher) kein bedarf, würde ki aber dafür nutzen',
]

def is_non_use_only(val):
    if pd.isna(val) or str(val).strip() == '' or str(val).strip().lower() == 'nan':
        return True
    return str(val).strip().lower() in non_use_answers

# Identify AI users who DO use AI for at least one of the 13 purposes
all_non_use_1_13 = pd.Series(True, index=df_use_ki.index)
for col in purpose_cols_1_13:
    all_non_use_1_13 &= df_use_ki[col].apply(is_non_use_only)

mask_use_for_any_purpose = ~all_non_use_1_13
df_use_ki_purposes = df_use_ki.loc[mask_use_for_any_purpose].copy()
n_use_ki_purposes = len(df_use_ki_purposes)

# G2Q1.1–G2Q1.7: consequences (readable labels from survey)
consequence_cols = [
    'ki_medizinische_hilfe_in_anspruch_genommen_haben_z_b_ar',   # G2Q1.1
    'ki_auf_einen_arztbesuch_verzichtet_haben',                   # G2Q1.2
    'ki_hat_der_austausch_mit_einem_ki_programm_fuer_gesundh',   # G2Q1.3
    'ki_hat_der_austausch_mit_einem_ki_programm_fuer_gesundh_2', # G2Q1.4
    'ki_hat_der_austausch_mit_einem_ki_programm_fuer_gesundh_3', # G2Q1.5
    'ki_hat_der_austausch_mit_einem_ki_programm_fuer_gesundh_4', # G2Q1.6
    'ki_sich_bei_emotionalen_themen_zusaetzlich_verunsichert',   # G2Q1.7
]
consequence_labels = {
    'ki_medizinische_hilfe_in_anspruch_genommen_haben_z_b_ar': 'G2Q1.1 — … seeking medical help (e.g., initiating a doctor\'s visit)?',
    'ki_auf_einen_arztbesuch_verzichtet_haben': 'G2Q1.2 — … foregoing a doctor\'s visit?',
    'ki_hat_der_austausch_mit_einem_ki_programm_fuer_gesundh': 'G2Q1.3 — … feeling better prepared for a doctor\'s consultation/appointment?',
    'ki_hat_der_austausch_mit_einem_ki_programm_fuer_gesundh_2': 'G2Q1.4 — … carrying out self-treatment recommended by the AI (e.g. home remedies, OTC meds, supplements)?',
    'ki_hat_der_austausch_mit_einem_ki_programm_fuer_gesundh_3': 'G2Q1.5 — … implementing a lifestyle change recommended by the AI (e.g. diet, exercise, sleep)?',
    'ki_hat_der_austausch_mit_einem_ki_programm_fuer_gesundh_4': 'G2Q1.6 — … feeling calmer about emotional topics (e.g. reduction of worries)?',
    'ki_sich_bei_emotionalen_themen_zusaetzlich_verunsichert': 'G2Q1.7 — … feeling (additionally) insecure about emotional topics?',
}

print('=' * 72)
print('Consequences of AI usage for health, psychological, or emotional topics')
print(f'Among AI users who use AI for at least one of the 13 G1Q4 purposes')
print(f'  N with any G1Q4 use: {n_use_ki_purposes} (out of {n_use_ki} AI users)')
print('=' * 72)

for col in consequence_cols:
    if col not in df_use_ki_purposes.columns:
        print(f"\n  [{consequence_labels.get(col, col)}] — column not found, skipped")
        continue
    s = df_use_ki_purposes[col].astype(str).str.strip()
    counts = s.value_counts(dropna=False)
    valid = s.replace('', np.nan).replace('nan', np.nan).dropna()
    pct_series = valid.value_counts(normalize=True) * 100 if len(valid) > 0 else pd.Series(dtype=float)
    label = consequence_labels.get(col, col)
    print(f'\n  {label}')
    print("  {'Answer':<50} {'N':>8}  {'%':>8}")
    print('  ' + '-' * 68)
    for val, count in counts.items():
        if pd.isna(val) or str(val).strip() == '' or str(val) == 'nan':
            print(f"  {'(missing)':<50} {count:>8}")
        else:
            pct = pct_series.get(val, 0.0)
            snippet = (val[:48] + '..') if len(val) > 50 else val
            print(f'  {snippet:<50} {count:>8}  {pct:>7.1f}%')
    print(f"  {'Total':<50} {len(df_use_ki_purposes):>8}")


Consequences of AI usage for health, psychological, or emotional topics
Among AI users who use AI for at least one of the 13 G1Q4 purposes
  N with any G1Q4 use: 9507 (out of 15058 AI users)

  G2Q1.1 — … seeking medical help (e.g., initiating a doctor's visit)?
  {'Answer':<50} {'N':>8}  {'%':>8}
  --------------------------------------------------------------------
  Nein                                                   7819     83.5%
  Ja                                                     1541     16.5%
  (missing)                                               147
  Total                                                  9507

  G2Q1.2 — … foregoing a doctor's visit?
  {'Answer':<50} {'N':>8}  {'%':>8}
  --------------------------------------------------------------------
  Nein                                                   8325     89.1%
  Ja                                                     1022     10.9%
  (missing)                                               160
  Total

### Side effects (all AI users)


In [36]:
# Side effects of AI usage — among all AI users (no G1Q4 purpose filter)
# Variables:
#   G2Q5.1.  -> ki_mein_verhaeltnis_zu_meinen_freunden_ist_seit_meiner
#   G2Q6.x   -> ki_ai_dependency
#   G2Q7.x   -> ki_decision_difficulty

import numpy as np
import pandas as pd

ki_use_col = 'ki_nutzen_sie_ki_programme_wie_z_b_chatgpt_gemini_claud'
mask_use_ki = df[ki_use_col].astype(str).str.strip().str.lower() == 'ja'
df_use_ki = df.loc[mask_use_ki].copy()
n_use_ki = len(df_use_ki)

side_effect_cols = [
    'ki_mein_verhaeltnis_zu_meinen_freunden_ist_seit_meiner',  # G2Q5.1.
    'ki_ai_dependency',                                        # G2Q6.x
    'ki_decision_difficulty',                                  # G2Q7.x
]

side_effect_labels = {
    'ki_mein_verhaeltnis_zu_meinen_freunden_ist_seit_meiner':
        'G2Q5.1 — My relationship with my friends since I started using AI programs is …',
    'ki_ai_dependency':
        'G2Q6 — I feel dependent on AI programs',
    'ki_decision_difficulty':
        'G2Q7 — Since I started using AI programs, it has become harder for me to make decisions on my own',
}

# Binary coding for “any side effect” (same as G1Q4-filtered block / KI_Regression1)
SOCIAL_YES_NUM = {-1, -2, -3}
YES_VALUES_TEXT = {
    'trifft ein wenig zu',
    'trifft teilweise zu',
    'trifft völlig zu',
    'trifft voellig zu',
}

def to_side_effect_yes(v, col):
    if pd.isna(v):
        return np.nan
    s = str(v).strip()
    if s == '' or s.lower() == 'nan':
        return np.nan
    if col == 'ki_mein_verhaeltnis_zu_meinen_freunden_ist_seit_meiner':
        num = pd.to_numeric(s, errors='coerce')
        if pd.notna(num) and int(num) in SOCIAL_YES_NUM:
            return 1
        return 0
    if s.lower() in YES_VALUES_TEXT:
        return 1
    return 0

print("=" * 72)
print("Side effects of AI usage (relationship, dependency, decision difficulty)")
print("Among all AI users (no filter on G1Q4 health-related purposes)")
print(f"  N AI users: {n_use_ki}")
print("=" * 72)

for col in side_effect_cols:
    if col not in df_use_ki.columns:
        print(f"\n  [{side_effect_labels.get(col, col)}] — column not found, skipped")
        continue
    s = df_use_ki[col].astype(str).str.strip()
    counts = s.value_counts(dropna=False)
    valid = s.replace('', np.nan).replace('nan', np.nan).dropna()
    pct_series = valid.value_counts(normalize=True) * 100 if len(valid) > 0 else pd.Series(dtype=float)
    label = side_effect_labels.get(col, col)
    print(f"\n  {label}")
    print("  {'Answer':<50} {'N':>8}  {'%':>8}")
    print("  " + "-" * 68)
    for val, count in counts.items():
        if pd.isna(val) or str(val).strip() == '' or str(val) == 'nan':
            print(f"  {'(missing)':<50} {count:>8}")
        else:
            pct = pct_series.get(val, 0.0)
            snippet = (val[:48] + '..') if len(val) > 50 else val
            print(f"  {snippet:<50} {count:>8}  {pct:>7.1f}%")
    print(f"  {'Total':<50} {len(df_use_ki):>8}")

# --- Person-level: ≥1 side effect (same definition as before) ---
yes_cols = []
for col in side_effect_cols:
    if col not in df_use_ki.columns:
        continue
    ycol = f'{col}_yes'
    df_use_ki[ycol] = df_use_ki[col].apply(lambda x, c=col: to_side_effect_yes(x, c))
    yes_cols.append(ycol)

if yes_cols:
    answered_any = df_use_ki[yes_cols].notna().any(axis=1)
    any_yes = df_use_ki[yes_cols].eq(1).any(axis=1)
    df_use_ki['side_effect_any'] = np.where(
        ~answered_any, np.nan, np.where(any_yes, 1, 0)
    )
    n_valid = int(df_use_ki['side_effect_any'].notna().sum())
    n_at_least_one = int((df_use_ki['side_effect_any'] == 1).sum())
    n_none = int((df_use_ki['side_effect_any'] == 0).sum())
    n_all_missing = int(df_use_ki['side_effect_any'].isna().sum())

    print("\n" + "=" * 72)
    print("Person-level summary (binary side-effect coding)")
    print(f"  N AI users:                            {len(df_use_ki):>8}")
    print(f"  With valid side_effect_any (answered ≥1 item): {n_valid:>8}")
    print(f"  Reported ≥1 side effect (side_effect_any = 1):   {n_at_least_one:>8}")
    if n_valid > 0:
        print(f"    → {100 * n_at_least_one / n_valid:.1f}% of those with ≥1 answered item")
    print(f"  Answered but no side effect ( = 0):   {n_none:>8}")
    print(f"  All three side-effect items missing:   {n_all_missing:>8}")
    print("=" * 72)

Side effects of AI usage (relationship, dependency, decision difficulty)
Among all AI users (no filter on G1Q4 health-related purposes)
  N AI users: 15058

  G2Q5.1 — My relationship with my friends since I started using AI programs is …
  {'Answer':<50} {'N':>8}  {'%':>8}
  --------------------------------------------------------------------
  0 (Unverändert)                                       14356     98.4%
  (missing)                                               470
  +1                                                      124      0.9%
  +2                                                       37      0.3%
  -1                                                       35      0.2%
  +3 (Viel besser)                                         20      0.1%
  -3 (Viel schlechter)                                      8      0.1%
  -2                                                        8      0.1%
  Total                                                 15058

  G2Q6 — I feel dependent

### Side effects by diagnosis status

Comparison of AI users with versus without a self-reported mental health diagnosis.


In [41]:
# Side effects of AI usage — compare AI users WITH vs WITHOUT mental health diagnosis
# Plus: OR for AI use by diagnosis (dx vs no-dx)

import numpy as np
import pandas as pd

ki_use_col = 'ki_nutzen_sie_ki_programme_wie_z_b_chatgpt_gemini_claud'
diagnose_selbst_col = 'Diagnose_selbst_dich'

# --- Masks (align to your coding) ---
mask_ai_yes = df[ki_use_col].astype(str).str.strip().str.lower().eq('ja')

# Diagnosis gate: you used 'a' as "yes diagnosis"
dx_raw = df[diagnose_selbst_col].astype(str).str.strip().str.lower()
mask_dx_yes = dx_raw.eq('a')

# Define "no diagnosis" as explicit 'b' if that exists in your data.
# If your dataset uses something else for "no", adjust here.
mask_dx_no = dx_raw.eq('b')

# --- Side-effect items ---
side_effect_cols = [
    'ki_mein_verhaeltnis_zu_meinen_freunden_ist_seit_meiner',  # G2Q5.1.
    'ki_ai_dependency',                                        # G2Q6.x
    'ki_decision_difficulty',                                  # G2Q7.x
]

side_effect_labels = {
    'ki_mein_verhaeltnis_zu_meinen_freunden_ist_seit_meiner':
        'G2Q5.1 — Relationship with friends since using AI programs is …',
    'ki_ai_dependency':
        'G2Q6 — I feel dependent on AI programs',
    'ki_decision_difficulty':
        'G2Q7 — Harder to make decisions on my own since using AI programs',
}

# Binary coding for “any side effect” (your exact logic)
SOCIAL_YES_NUM = {-1, -2, -3}
YES_VALUES_TEXT = {
    'trifft ein wenig zu',
    'trifft teilweise zu',
    'trifft völlig zu',
    'trifft voellig zu',
}

def to_side_effect_yes(v, col):
    if pd.isna(v):
        return np.nan
    s = str(v).strip()
    if s == '' or s.lower() == 'nan':
        return np.nan

    if col == 'ki_mein_verhaeltnis_zu_meinen_freunden_ist_seit_meiner':
        num = pd.to_numeric(s, errors='coerce')
        if pd.notna(num) and int(num) in SOCIAL_YES_NUM:
            return 1
        return 0

    if s.lower() in YES_VALUES_TEXT:
        return 1
    return 0

def add_side_effect_any(dsub: pd.DataFrame) -> pd.DataFrame:
    dsub = dsub.copy()
    yes_cols = []
    for col in side_effect_cols:
        if col not in dsub.columns:
            continue
        ycol = f"{col}_yes"
        dsub[ycol] = dsub[col].apply(lambda x, c=col: to_side_effect_yes(x, c))
        yes_cols.append(ycol)

    if not yes_cols:
        dsub["side_effect_any"] = np.nan
        return dsub

    answered_any = dsub[yes_cols].notna().any(axis=1)
    any_yes = dsub[yes_cols].eq(1).any(axis=1)
    dsub["side_effect_any"] = np.where(~answered_any, np.nan, np.where(any_yes, 1, 0))
    return dsub

def odds_ratio_2x2(a, b, c, d):
    """
    2x2 table:
                 outcome=1   outcome=0
    group=1          a          b
    group=0          c          d
    Returns OR and Wald 95% CI on log(OR). (No continuity correction.)
    """
    a, b, c, d = map(float, [a, b, c, d])
    if min(a, b, c, d) <= 0:
        return np.nan, (np.nan, np.nan)

    or_ = (a * d) / (b * c)
    se = np.sqrt(1/a + 1/b + 1/c + 1/d)
    lo = np.exp(np.log(or_) - 1.96 * se)
    hi = np.exp(np.log(or_) + 1.96 * se)
    return or_, (lo, hi)

# ======================================================================================
# A) Side effects among AI users: dx yes vs dx no (same summaries, plus OR for side_effect_any)
# ======================================================================================

df_ai_dx = df.loc[mask_ai_yes & mask_dx_yes].copy()
df_ai_no = df.loc[mask_ai_yes & mask_dx_no].copy()

df_ai_dx = add_side_effect_any(df_ai_dx)
df_ai_no = add_side_effect_any(df_ai_no)

def side_effect_summary(dsub, title):
    print("\n" + "=" * 72)
    print(title)
    print(f"N in subgroup: {len(dsub)}")
    if "side_effect_any" not in dsub.columns:
        print("side_effect_any not available (no side-effect columns found).")
        print("=" * 72)
        return

    n_valid = int(dsub["side_effect_any"].notna().sum())
    n_yes = int((dsub["side_effect_any"] == 1).sum())
    n_no = int((dsub["side_effect_any"] == 0).sum())
    n_all_missing = int(dsub["side_effect_any"].isna().sum())

    print(f"With valid side_effect_any (answered ≥1 item): {n_valid}")
    print(f"Reported ≥1 side effect (side_effect_any=1):  {n_yes}")
    if n_valid > 0:
        print(f"  → {100*n_yes/n_valid:.1f}% (among those with ≥1 answered item)")
    print(f"Answered but no side effect (side_effect_any=0): {n_no}")
    print(f"All three items missing: {n_all_missing}")
    print("=" * 72)

side_effect_summary(df_ai_dx, "Side effects among AI users WITH diagnosis (Diagnose_selbst_dich == 'a')")
side_effect_summary(df_ai_no, "Side effects among AI users WITHOUT diagnosis (Diagnose_selbst_dich == 'b')")

# OR: side_effect_any (1 vs 0) comparing dx_yes vs dx_no, restricted to those with valid side_effect_any
dx_valid = df_ai_dx.loc[df_ai_dx["side_effect_any"].notna(), "side_effect_any"]
no_valid = df_ai_no.loc[df_ai_no["side_effect_any"].notna(), "side_effect_any"]

a = int((dx_valid == 1).sum())  # dx_yes & side_effect=1
b = int((dx_valid == 0).sum())  # dx_yes & side_effect=0
c = int((no_valid == 1).sum())  # dx_no  & side_effect=1
d = int((no_valid == 0).sum())  # dx_no  & side_effect=0

or_sidefx, (lo_sidefx, hi_sidefx) = odds_ratio_2x2(a, b, c, d)

print("\n" + "=" * 72)
print("OR for reporting ≥1 side effect (dx vs no-dx), among AI users with valid side-effect data")
print(f"2x2 counts (dx_yes vs dx_no):")
print(f"  dx_yes: side_effect=1: {a}, side_effect=0: {b}")
print(f"   dx_no: side_effect=1: {c}, side_effect=0: {d}")
print(f"  OR = {or_sidefx:.3f}  (95% CI {lo_sidefx:.3f} – {hi_sidefx:.3f})")
print("=" * 72)

# ======================================================================================
# B) OR for AI use (Yes vs No) comparing dx_yes vs dx_no (full sample with dx explicitly coded)
# ======================================================================================

# Define AI "No" (mirror your earlier approach; adjust if needed)
mask_ai_no = df[ki_use_col].astype(str).str.strip().str.lower().eq('nein')

# Restrict to people with explicit dx yes/no and explicit AI yes/no
mask_dx_known = mask_dx_yes | mask_dx_no
mask_ai_known = mask_ai_yes | mask_ai_no

d2 = df.loc[mask_dx_known & mask_ai_known].copy()

dx_yes = d2.loc[mask_dx_yes.loc[d2.index]]
dx_no_ = d2.loc[mask_dx_no.loc[d2.index]]

a2 = int(mask_ai_yes.loc[dx_yes.index].sum())  # dx_yes & AI=Yes
b2 = int(mask_ai_no.loc[dx_yes.index].sum())   # dx_yes & AI=No
c2 = int(mask_ai_yes.loc[dx_no_.index].sum())  # dx_no  & AI=Yes
d2_ = int(mask_ai_no.loc[dx_no_.index].sum())  # dx_no  & AI=No

or_aiuse, (lo_aiuse, hi_aiuse) = odds_ratio_2x2(a2, b2, c2, d2_)

print("\n" + "=" * 72)
print("OR for being an AI user (Ja vs Nein) comparing dx_yes vs dx_no (explicit dx only)")
print(f"2x2 counts (dx_yes vs dx_no):")
print(f"  dx_yes: AI=Yes: {a2}, AI=No: {b2}")
print(f"   dx_no: AI=Yes: {c2}, AI=No: {d2_}")
print(f"  OR = {or_aiuse:.3f}  (95% CI {lo_aiuse:.3f} – {hi_aiuse:.3f})")
print("=" * 72)


Side effects among AI users WITH diagnosis (Diagnose_selbst_dich == 'a')
N in subgroup: 1043
With valid side_effect_any (answered ≥1 item): 1035
Reported ≥1 side effect (side_effect_any=1):  124
  → 12.0% (among those with ≥1 answered item)
Answered but no side effect (side_effect_any=0): 911
All three items missing: 8

Side effects among AI users WITHOUT diagnosis (Diagnose_selbst_dich == 'b')
N in subgroup: 4627
With valid side_effect_any (answered ≥1 item): 4591
Reported ≥1 side effect (side_effect_any=1):  435
  → 9.5% (among those with ≥1 answered item)
Answered but no side effect (side_effect_any=0): 4156
All three items missing: 36

OR for reporting ≥1 side effect (dx vs no-dx), among AI users with valid side-effect data
2x2 counts (dx_yes vs dx_no):
  dx_yes: side_effect=1: 124, side_effect=0: 911
   dx_no: side_effect=1: 435, side_effect=0: 4156
  OR = 1.300  (95% CI 1.052 – 1.608)

OR for being an AI user (Ja vs Nein) comparing dx_yes vs dx_no (explicit dx only)
2x2 counts (

### Side effects (health-related AI use only)


In [38]:
# Side effects of AI usage — among AI users who use AI for at least one of the 13 G1Q4 purposes
# Variables:
#   G2Q5.1.  -> ki_mein_verhaeltnis_zu_meinen_freunden_ist_seit_meiner
#   G2Q6.x   -> ki_ai_dependency
#   G2Q7.x   -> ki_decision_difficulty

ki_use_col = 'ki_nutzen_sie_ki_programme_wie_z_b_chatgpt_gemini_claud'
mask_use_ki = df[ki_use_col].astype(str).str.strip().str.lower() == 'ja'
df_use_ki = df.loc[mask_use_ki].copy()
n_use_ki = len(df_use_ki)

# G1Q4.1–G1Q4.13: all medical + psychological/emotional purposes
purpose_cols_1_13 = [
    'ki_wie_haeufig_nutzen_sie_ki_programme_fuer_die_folgend',      # G1Q4.1
    'ki_wie_haeufig_nutzen_sie_ki_programme_fuer_die_folgend_2',    # G1Q4.2
    'ki_wie_haeufig_nutzen_sie_ki_programme_fuer_die_folgend_3',    # G1Q4.3
    'ki_bei_aengsten_und_sorgen',                                   # G1Q4.4
    'ki_bei_selbstzweifeln_oder_niedergeschlagenheit',              # G1Q4.5
    'ki_bei_einsamkeit_z_b_um_jemanden_zum_reden_zu_haben_we',      # G1Q4.6
    'ki_bei_zwischenmenschlichen_konflikten',                        # G1Q4.7
    'ki_bei_unsicherheiten_bei_entscheidungen_um_meine_gedan',      # G1Q4.8
    'ki_bei_trauer_oder_verlust',                                   # G1Q4.9
    'ki_wenn_grosse_lebensveraenderungen_anstehen',                  # G1Q4.10
    'ki_wie_haeufig_nutzen_sie_ki_programme_fuer_die_folgend_4',    # G1Q4.11
    'ki_wie_haeufig_nutzen_sie_ki_programme_fuer_die_folgend_5',    # G1Q4.12
    'ki_fuer_sonstige_gesundheits_psychische_oder_emotionale',      # G1Q4.13
]
purpose_cols_1_13 = [c for c in purpose_cols_1_13 if c in df_use_ki.columns]

non_use_answers = [
    'hierfür nutze ich keine ki',
    '(bisher) kein bedarf, würde ki aber dafür nutzen',
]

def is_non_use_only(val):
    if pd.isna(val) or str(val).strip() == '' or str(val).strip().lower() == 'nan':
        return True
    return str(val).strip().lower() in non_use_answers

# AI users who DO use AI for at least one of the 13 purposes
all_non_use_1_13 = pd.Series(True, index=df_use_ki.index)
for col in purpose_cols_1_13:
    all_non_use_1_13 &= df_use_ki[col].apply(is_non_use_only)

mask_use_for_any_purpose = ~all_non_use_1_13
df_use_ki_purposes = df_use_ki.loc[mask_use_for_any_purpose].copy()
n_use_ki_purposes = len(df_use_ki_purposes)

side_effect_cols = [
    'ki_mein_verhaeltnis_zu_meinen_freunden_ist_seit_meiner',  # G2Q5.1.
    'ki_ai_dependency',                                        # G2Q6.x
    'ki_decision_difficulty',                                  # G2Q7.x
]

side_effect_labels = {
    'ki_mein_verhaeltnis_zu_meinen_freunden_ist_seit_meiner':
        'G2Q5.1 — My relationship with my friends since I started using AI programs is …',
    'ki_ai_dependency':
        'G2Q6 — I feel dependent on AI programs',
    'ki_decision_difficulty':
        'G2Q7 — Since I started using AI programs, it has become harder for me to make decisions on my own',
}

# --- Binary coding for “any side effect” (same logic as KI_Regression1 / mediation) ---
SOCIAL_YES_NUM = {-1, -2, -3}
YES_VALUES_TEXT = {
    'trifft ein wenig zu',
    'trifft teilweise zu',
    'trifft völlig zu',
    'trifft voellig zu',
}

def to_side_effect_yes(v, col):
    if pd.isna(v):
        return np.nan
    s = str(v).strip()
    if s == '' or s.lower() == 'nan':
        return np.nan
    if col == 'ki_mein_verhaeltnis_zu_meinen_freunden_ist_seit_meiner':
        num = pd.to_numeric(s, errors='coerce')
        if pd.notna(num) and int(num) in SOCIAL_YES_NUM:
            return 1
        return 0
    if s.lower() in YES_VALUES_TEXT:
        return 1
    return 0

print("=" * 72)
print("Side effects of AI usage (relationship, dependency, decision difficulty)")
print("Among AI users who use AI for at least one of the 13 G1Q4 purposes")
print(f"  N with any G1Q4 use: {n_use_ki_purposes} (out of {n_use_ki} AI users)")
print("=" * 72)

for col in side_effect_cols:
    if col not in df_use_ki_purposes.columns:
        print(f"\n  [{side_effect_labels.get(col, col)}] — column not found, skipped")
        continue
    s = df_use_ki_purposes[col].astype(str).str.strip()
    counts = s.value_counts(dropna=False)
    valid = s.replace('', np.nan).replace('nan', np.nan).dropna()
    pct_series = valid.value_counts(normalize=True) * 100 if len(valid) > 0 else pd.Series(dtype=float)
    label = side_effect_labels.get(col, col)
    print(f"\n  {label}")
    print("  {'Answer':<50} {'N':>8}  {'%':>8}")
    print("  " + "-" * 68)
    for val, count in counts.items():
        if pd.isna(val) or str(val).strip() == '' or str(val) == 'nan':
            print(f"  {'(missing)':<50} {count:>8}")
        else:
            pct = pct_series.get(val, 0.0)
            snippet = (val[:48] + '..') if len(val) > 50 else val
            print(f"  {snippet:<50} {count:>8}  {pct:>7.1f}%")
    print(f"  {'Total':<50} {len(df_use_ki_purposes):>8}")

# Person-level: at least one side effect (binary rules above)
yes_cols = []
for col in side_effect_cols:
    if col not in df_use_ki_purposes.columns:
        continue
    ycol = f'{col}_yes'
    df_use_ki_purposes[ycol] = df_use_ki_purposes[col].apply(lambda x, c=col: to_side_effect_yes(x, c))
    yes_cols.append(ycol)

if yes_cols:
    answered_any = df_use_ki_purposes[yes_cols].notna().any(axis=1)
    any_yes = df_use_ki_purposes[yes_cols].eq(1).any(axis=1)
    # 1 = ≥1 side effect; 0 = answered ≥1 item but none “yes”; NaN = all three missing
    df_use_ki_purposes['side_effect_any'] = np.where(
        ~answered_any, np.nan, np.where(any_yes, 1, 0)
    )
    n_valid = int(df_use_ki_purposes['side_effect_any'].notna().sum())
    n_at_least_one = int((df_use_ki_purposes['side_effect_any'] == 1).sum())
    n_none = int((df_use_ki_purposes['side_effect_any'] == 0).sum())
    n_all_missing = int(df_use_ki_purposes['side_effect_any'].isna().sum())

    print("\n" + "=" * 72)
    print("Person-level summary (binary side-effect coding; see SOCIAL_YES_NUM / YES_VALUES_TEXT)")
    print(f"  N in this subgroup:                    {len(df_use_ki_purposes):>8}")
    print(f"  With valid side_effect_any (answered ≥1 item): {n_valid:>8}")
    print(f"  Reported ≥1 side effect (side_effect_any = 1):   {n_at_least_one:>8}")
    if n_valid > 0:
        print(f"    → {100 * n_at_least_one / n_valid:.1f}% of those with ≥1 answered item")
    print(f"  Answered but no side effect ( = 0):   {n_none:>8}")
    print(f"  All three side-effect items missing:   {n_all_missing:>8}")
    print("=" * 72)

Side effects of AI usage (relationship, dependency, decision difficulty)
Among AI users who use AI for at least one of the 13 G1Q4 purposes
  N with any G1Q4 use: 9507 (out of 15058 AI users)

  G2Q5.1 — My relationship with my friends since I started using AI programs is …
  {'Answer':<50} {'N':>8}  {'%':>8}
  --------------------------------------------------------------------
  0 (Unverändert)                                        9112     97.8%
  (missing)                                               187
  +1                                                      116      1.2%
  +2                                                       35      0.4%
  -1                                                       30      0.3%
  +3 (Viel besser)                                         16      0.2%
  -2                                                        6      0.1%
  -3 (Viel schlechter)                                      5      0.1%
  Total                                             

### Side effects (health-related AI use), stratified by diagnosis


In [43]:
# Compare side effects among HEALTH-RELATED LLM users (≥1 G1Q4 purpose)
# Diagnosis yes (a) vs no (b), with ORs for ≥1 side effect

import numpy as np
import pandas as pd

ki_use_col = "ki_nutzen_sie_ki_programme_wie_z_b_chatgpt_gemini_claud"
diagnose_col = "Diagnose_selbst_dich"

# --- AI users ---
mask_ai_yes = df[ki_use_col].astype(str).str.strip().str.lower().eq("ja")

# --- Diagnosis groups (adjust if your "no" code differs) ---
dx_raw = df[diagnose_col].astype(str).str.strip().str.lower()
mask_dx_yes = dx_raw.eq("a")
mask_dx_no  = dx_raw.eq("b")  # "no diagnosis"

# --- G1Q4.1–G1Q4.13 purposes (health + psychological/emotional) ---
purpose_cols_1_13 = [
    "ki_wie_haeufig_nutzen_sie_ki_programme_fuer_die_folgend",      # G1Q4.1
    "ki_wie_haeufig_nutzen_sie_ki_programme_fuer_die_folgend_2",    # G1Q4.2
    "ki_wie_haeufig_nutzen_sie_ki_programme_fuer_die_folgend_3",    # G1Q4.3
    "ki_bei_aengsten_und_sorgen",                                   # G1Q4.4
    "ki_bei_selbstzweifeln_oder_niedergeschlagenheit",              # G1Q4.5
    "ki_bei_einsamkeit_z_b_um_jemanden_zum_reden_zu_haben_we",      # G1Q4.6
    "ki_bei_zwischenmenschlichen_konflikten",                        # G1Q4.7
    "ki_bei_unsicherheiten_bei_entscheidungen_um_meine_gedan",      # G1Q4.8
    "ki_bei_trauer_oder_verlust",                                   # G1Q4.9
    "ki_wenn_grosse_lebensveraenderungen_anstehen",                  # G1Q4.10
    "ki_wie_haeufig_nutzen_sie_ki_programme_fuer_die_folgend_4",    # G1Q4.11
    "ki_wie_haeufig_nutzen_sie_ki_programme_fuer_die_folgend_5",    # G1Q4.12
    "ki_fuer_sonstige_gesundheits_psychische_oder_emotionale",      # G1Q4.13
]
purpose_cols_1_13 = [c for c in purpose_cols_1_13 if c in df.columns]

non_use_answers = [
    "hierfür nutze ich keine ki",
    "(bisher) kein bedarf, würde ki aber dafür nutzen",
]

def is_non_use_only(val):
    # NOTE: this mirrors your original logic (missing -> treated as non-use)
    if pd.isna(val) or str(val).strip() == "" or str(val).strip().lower() == "nan":
        return True
    return str(val).strip().lower() in non_use_answers

def has_any_g1q4_use(dsub: pd.DataFrame) -> pd.Series:
    if not purpose_cols_1_13:
        return pd.Series(False, index=dsub.index)
    all_non_use = pd.Series(True, index=dsub.index)
    for col in purpose_cols_1_13:
        all_non_use &= dsub[col].apply(is_non_use_only)
    return ~all_non_use

# --- Side-effect items ---
side_effect_cols = [
    "ki_mein_verhaeltnis_zu_meinen_freunden_ist_seit_meiner",  # G2Q5.1
    "ki_ai_dependency",                                        # G2Q6
    "ki_decision_difficulty",                                  # G2Q7
]
side_effect_cols = [c for c in side_effect_cols if c in df.columns]

SOCIAL_YES_NUM = {-1, -2, -3}
YES_VALUES_TEXT = {
    "trifft ein wenig zu",
    "trifft teilweise zu",
    "trifft völlig zu",
    "trifft voellig zu",
}

def to_side_effect_yes(v, col):
    if pd.isna(v):
        return np.nan
    s = str(v).strip()
    if s == "" or s.lower() == "nan":
        return np.nan

    if col == "ki_mein_verhaeltnis_zu_meinen_freunden_ist_seit_meiner":
        num = pd.to_numeric(s, errors="coerce")
        if pd.notna(num) and int(num) in SOCIAL_YES_NUM:
            return 1
        return 0

    if s.lower() in YES_VALUES_TEXT:
        return 1
    return 0

def add_side_effect_any(dsub: pd.DataFrame) -> pd.DataFrame:
    dsub = dsub.copy()
    yes_cols = []
    for col in side_effect_cols:
        ycol = f"{col}_yes"
        dsub[ycol] = dsub[col].apply(lambda x, c=col: to_side_effect_yes(x, c))
        yes_cols.append(ycol)

    if not yes_cols:
        dsub["side_effect_any"] = np.nan
        return dsub

    answered_any = dsub[yes_cols].notna().any(axis=1)
    any_yes = dsub[yes_cols].eq(1).any(axis=1)
    dsub["side_effect_any"] = np.where(~answered_any, np.nan, np.where(any_yes, 1, 0))
    return dsub

def odds_ratio_2x2(a, b, c, d):
    # dx_yes is group=1; dx_no is group=0
    a, b, c, d = map(float, [a, b, c, d])
    if min(a, b, c, d) <= 0:
        return np.nan, (np.nan, np.nan)
    or_ = (a * d) / (b * c)
    se = np.sqrt(1/a + 1/b + 1/c + 1/d)
    lo = np.exp(np.log(or_) - 1.96 * se)
    hi = np.exp(np.log(or_) + 1.96 * se)
    return or_, (lo, hi)

# ======================================================================================
# Build HEALTH-RELATED AI-user subgroup in each dx group
# ======================================================================================

df_ai_dx = df.loc[mask_ai_yes & mask_dx_yes].copy()
df_ai_no = df.loc[mask_ai_yes & mask_dx_no].copy()

df_ai_dx = df_ai_dx.loc[has_any_g1q4_use(df_ai_dx)].copy()
df_ai_no = df_ai_no.loc[has_any_g1q4_use(df_ai_no)].copy()

df_ai_dx = add_side_effect_any(df_ai_dx)
df_ai_no = add_side_effect_any(df_ai_no)

def summarize(dsub, label):
    n = len(dsub)
    n_valid = int(dsub["side_effect_any"].notna().sum())
    n_yes = int((dsub["side_effect_any"] == 1).sum())
    n_no  = int((dsub["side_effect_any"] == 0).sum())
    n_miss = int(dsub["side_effect_any"].isna().sum())
    pct = (100 * n_yes / n_valid) if n_valid else np.nan
    print(f"\n{label}")
    print(f"  N subgroup: {n}")
    print(f"  Valid side_effect_any: {n_valid}")
    print(f"  side_effect_any=1: {n_yes} ({pct:.1f}%)")
    print(f"  side_effect_any=0: {n_no}")
    print(f"  All missing: {n_miss}")
    return n_yes, n_no

print("=" * 72)
print("Side effects among HEALTH-RELATED LLM users (≥1 G1Q4 purpose)")
print("Comparing diagnosis yes ('a') vs no ('b')")
print("=" * 72)

a_yes, b_no = summarize(df_ai_dx, "Dx yes (a): AI Yes + ≥1 G1Q4 purpose")
c_yes, d_no = summarize(df_ai_no, "Dx no  (b): AI Yes + ≥1 G1Q4 purpose")

# OR among those with valid side_effect_any (answered ≥1 side-effect item)
dx_valid = df_ai_dx.loc[df_ai_dx["side_effect_any"].notna(), "side_effect_any"]
no_valid = df_ai_no.loc[df_ai_no["side_effect_any"].notna(), "side_effect_any"]

a = int((dx_valid == 1).sum())
b = int((dx_valid == 0).sum())
c = int((no_valid == 1).sum())
d = int((no_valid == 0).sum())

or_sidefx, (lo, hi) = odds_ratio_2x2(a, b, c, d)

print("\n" + "=" * 72)
print("OR for reporting ≥1 side effect (dx vs no-dx) among HEALTH-RELATED LLM users")
print(f"2x2 counts (valid side_effect_any only):")
print(f"  dx_yes: side_effect=1: {a}, side_effect=0: {b}")
print(f"   dx_no: side_effect=1: {c}, side_effect=0: {d}")
print(f"  OR = {or_sidefx:.3f}  (95% CI {lo:.3f} – {hi:.3f})")
print("=" * 72)

Side effects among HEALTH-RELATED LLM users (≥1 G1Q4 purpose)
Comparing diagnosis yes ('a') vs no ('b')

Dx yes (a): AI Yes + ≥1 G1Q4 purpose
  N subgroup: 727
  Valid side_effect_any: 720
  side_effect_any=1: 111 (15.4%)
  side_effect_any=0: 609
  All missing: 7

Dx no  (b): AI Yes + ≥1 G1Q4 purpose
  N subgroup: 2711
  Valid side_effect_any: 2685
  side_effect_any=1: 325 (12.1%)
  side_effect_any=0: 2360
  All missing: 26

OR for reporting ≥1 side effect (dx vs no-dx) among HEALTH-RELATED LLM users
2x2 counts (valid side_effect_any only):
  dx_yes: side_effect=1: 111, side_effect=0: 609
   dx_no: side_effect=1: 325, side_effect=0: 2360
  OR = 1.324  (95% CI 1.048 – 1.671)


## Trust in AI answers


In [40]:
# Evaluation of answer quality: G2Q11.1
# "Ich halte die Antworten, die ich von einem KI-Programm zu Gesundheits-, psychischen und emotionalen Themen bekomme, für ..."
# Variable: ki_ich_halte_die_antworten_die_ich_von_einem_ki_program

from scipy import stats

ki_use_col = 'ki_nutzen_sie_ki_programme_wie_z_b_chatgpt_gemini_claud'
mask_use_ki = df[ki_use_col].astype(str).str.strip().str.lower() == 'ja'
df_use_ki = df.loc[mask_use_ki].copy()
n_use_ki = len(df_use_ki)

# G1Q4.1–G1Q4.13: all health / psych / emotional purposes
purpose_cols_1_13 = [
    'ki_wie_haeufig_nutzen_sie_ki_programme_fuer_die_folgend',      # G1Q4.1
    'ki_wie_haeufig_nutzen_sie_ki_programme_fuer_die_folgend_2',    # G1Q4.2
    'ki_wie_haeufig_nutzen_sie_ki_programme_fuer_die_folgend_3',    # G1Q4.3
    'ki_bei_aengsten_und_sorgen',                                   # G1Q4.4
    'ki_bei_selbstzweifeln_oder_niedergeschlagenheit',              # G1Q4.5
    'ki_bei_einsamkeit_z_b_um_jemanden_zum_reden_zu_haben_we',      # G1Q4.6
    'ki_bei_zwischenmenschlichen_konflikten',                        # G1Q4.7
    'ki_bei_unsicherheiten_bei_entscheidungen_um_meine_gedan',      # G1Q4.8
    'ki_bei_trauer_oder_verlust',                                   # G1Q4.9
    'ki_wenn_grosse_lebensveraenderungen_anstehen',                  # G1Q4.10
    'ki_wie_haeufig_nutzen_sie_ki_programme_fuer_die_folgend_4',    # G1Q4.11
    'ki_wie_haeufig_nutzen_sie_ki_programme_fuer_die_folgend_5',    # G1Q4.12
    'ki_fuer_sonstige_gesundheits_psychische_oder_emotionale',      # G1Q4.13
]
purpose_cols_1_13 = [c for c in purpose_cols_1_13 if c in df_use_ki.columns]

non_use_answers = [
    'hierfür nutze ich keine ki',
    '(bisher) kein bedarf, würde ki aber dafür nutzen',
]

def is_non_use_only(val):
    if pd.isna(val) or str(val).strip() == '' or str(val).strip().lower() == 'nan':
        return True
    return str(val).strip().lower() in non_use_answers

# AI users who use AI for at least one of the 13 purposes
all_non_use_1_13 = pd.Series(True, index=df_use_ki.index)
for col in purpose_cols_1_13:
    all_non_use_1_13 &= df_use_ki[col].apply(is_non_use_only)

mask_any_g1q4_use = ~all_non_use_1_13
df_g1q4_users = df_use_ki.loc[mask_any_g1q4_use].copy()
n_g1q4_users = len(df_g1q4_users)

col_quality = 'ki_ich_halte_die_antworten_die_ich_von_einem_ki_program'

# Basic descriptives for G2Q11.1
s = df_g1q4_users[col_quality]
x = pd.to_numeric(s, errors='coerce').dropna()
n = len(x)
mean = x.mean()
median = x.median()
std = x.std(ddof=1)
sem = stats.sem(x, nan_policy='omit') if n > 1 else float('nan')
alpha = 0.05
if n > 1:
    ci_low, ci_high = stats.t.interval(1 - alpha, df=n-1, loc=mean, scale=sem)
else:
    ci_low = ci_high = float('nan')

quantiles = x.quantile([0.05, 0.25, 0.75, 0.95])

print("=" * 72)
print("G2Q11.1 — Perceived quality of AI answers")
print("Among AI users who use AI for at least one of the 13 G1Q4 purposes")
print(f"  N with any G1Q4 use: {n_g1q4_users} (out of {n_use_ki} AI users)")
print(f"  N with valid G2Q11.1 value: {n}")
print("=" * 72)
print(f"  Mean:   {mean:.3f}")
print(f"  Median: {median:.3f}")
print(f"  SD:     {std:.3f}")
print(f"  95% CI for the mean: [{ci_low:.3f}, {ci_high:.3f}]")
print("  Quantiles (5%, 25%, 75%, 95%):")
for q, v in quantiles.items():
    print(f"    {int(q*100):>3}%: {v:.3f}")



G2Q11.1 — Perceived quality of AI answers
Among AI users who use AI for at least one of the 13 G1Q4 purposes
  N with any G1Q4 use: 9507 (out of 15058 AI users)
  N with valid G2Q11.1 value: 8511
  Mean:   50.420
  Median: 50.000
  SD:     23.770
  95% CI for the mean: [49.915, 50.925]
  Quantiles (5%, 25%, 75%, 95%):
      5%: 10.000
     25%: 30.000
     75%: 70.000
     95%: 85.000
